In [ ]:
# In[1]:

import pandas as pd
import pytz

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs (they are expected to be in current working directory)
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_error = pd.read_csv('error_logs.csv')

# Helper to convert epoch seconds column to localized datetime (UTC+8)
def to_local_dt(series):
    # series assumed numeric seconds
    return pd.to_datetime(series, unit='s', utc=True).dt.tz_convert(tz)

# --- metric.csv summary ---
# total rows
metric_total_rows = df_metric.shape[0]
# timestamps
metric_min_ts = to_local_dt(df_metric['timestamp']).min()
metric_max_ts = to_local_dt(df_metric['timestamp']).max()
# unique cmdb_id
metric_unique_cmdb = sorted(df_metric['cmdb_id'].dropna().unique())
metric_unique_cmdb_count = len(metric_unique_cmdb)
metric_unique_cmdb_list = metric_unique_cmdb[:50]
# unique kpi_name count
metric_unique_kpi_count = df_metric['kpi_name'].nunique(dropna=True)
# kpi_name counts (top 50)
metric_kpi_counts = (
    df_metric['kpi_name']
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={'index': 'kpi_name', 'kpi_name': 'count'})
)
metric_kpi_counts_top50 = metric_kpi_counts.head(50)
# top 20 (cmdb_id, kpi_name) pairs by row count
metric_pair_counts = (
    df_metric.groupby(['cmdb_id', 'kpi_name'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)
metric_top_pairs = metric_pair_counts.head(20)

# Compose compact metric stats DataFrame (single-row)
metric_stats = pd.DataFrame([{
    'file': 'metric.csv',
    'total_rows': metric_total_rows,
    'min_timestamp_UTC+8': metric_min_ts,
    'max_timestamp_UTC+8': metric_max_ts,
    'unique_cmdb_count': metric_unique_cmdb_count,
    'unique_kpi_count': metric_unique_kpi_count,
    'sample_unique_cmdb_list_(first_50)': ','.join(map(str, metric_unique_cmdb_list))
}])

# --- trace.csv summary ---
trace_total_rows = df_trace.shape[0]
trace_min_ts = to_local_dt(df_trace['timestamp']).min()
trace_max_ts = to_local_dt(df_trace['timestamp']).max()
trace_unique_cmdb = sorted(df_trace['cmdb_id'].dropna().unique())
trace_unique_cmdb_count = len(trace_unique_cmdb)
trace_unique_cmdb_list = trace_unique_cmdb[:50]
trace_unique_name_count = df_trace['trace_name'].nunique(dropna=True)
trace_name_counts = (
    df_trace['trace_name']
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={'index': 'trace_name', 'trace_name': 'count'})
)
trace_name_counts_top50 = trace_name_counts.head(50)

trace_stats = pd.DataFrame([{
    'file': 'trace.csv',
    'total_rows': trace_total_rows,
    'min_timestamp_UTC+8': trace_min_ts,
    'max_timestamp_UTC+8': trace_max_ts,
    'unique_cmdb_count': trace_unique_cmdb_count,
    'unique_trace_name_count': trace_unique_name_count,
    'sample_unique_cmdb_list_(first_50)': ','.join(map(str, trace_unique_cmdb_list))
}])

# --- log.csv summary ---
log_total_rows = df_log.shape[0]
log_min_ts = to_local_dt(df_log['timestamp']).min()
log_max_ts = to_local_dt(df_log['timestamp']).max()
log_unique_cmdb = sorted(df_log['cmdb_id'].dropna().unique())
log_unique_cmdb_count = len(log_unique_cmdb)
log_unique_cmdb_list = log_unique_cmdb[:50]
log_unique_name_count = df_log['log_name'].nunique(dropna=True)
log_name_counts = (
    df_log['log_name']
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={'index': 'log_name', 'log_name': 'count'})
)
log_name_counts_top50 = log_name_counts.head(50)

log_stats = pd.DataFrame([{
    'file': 'log.csv',
    'total_rows': log_total_rows,
    'min_timestamp_UTC+8': log_min_ts,
    'max_timestamp_UTC+8': log_max_ts,
    'unique_cmdb_count': log_unique_cmdb_count,
    'unique_log_name_count': log_unique_name_count,
    'sample_unique_cmdb_list_(first_50)': ','.join(map(str, log_unique_cmdb_list))
}])

# --- error_logs.csv summary ---
err_total_rows = df_error.shape[0]
err_min_ts = to_local_dt(df_error['timestamp']).min()
err_max_ts = to_local_dt(df_error['timestamp']).max()
err_unique_cmdb = sorted(df_error['cmdb_id'].dropna().unique())
err_unique_cmdb_count = len(err_unique_cmdb)
err_unique_cmdb_list = err_unique_cmdb[:50]
# distinct messages count and top messages by occurrence
err_unique_message_count = df_error['message'].nunique(dropna=True)
err_message_counts = (
    df_error['message']
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={'index': 'message', 'message': 'count'})
)
err_message_counts_top50 = err_message_counts.head(50)

error_stats = pd.DataFrame([{
    'file': 'error_logs.csv',
    'total_rows': err_total_rows,
    'min_timestamp_UTC+8': err_min_ts,
    'max_timestamp_UTC+8': err_max_ts,
    'unique_cmdb_count': err_unique_cmdb_count,
    'distinct_message_count': err_unique_message_count,
    'sample_unique_cmdb_list_(first_50)': ','.join(map(str, err_unique_cmdb_list))
}])

# Final compact outputs (ordered)
metric_stats, metric_kpi_counts_top50, metric_top_pairs, trace_stats, trace_name_counts_top50, log_stats, log_name_counts_top50, error_stats, err_message_counts_top50

```
Out[1]:
```
summary = (
    "Summary of telemetry files (timestamps shown in UTC+8):\n\n"
    "metric.csv:\n"
    "- Total rows: 145,092\n"
    "- Time range: 2022-03-20 21:00:00+08 to 2022-03-20 21:29:00+08\n"
    "- Unique cmdb_id: 59 (sample first entries: adservice, adservice-0, adservice-1, adservice-2, ...)\n"
    "- Unique kpi_name count: 1,176\n"
    "- Top KPI names by occurrence (examples): several container.node-5 network/memory metrics (~1,050 occurrences each); many other KPIs around 990 occurrences.\n"
    "- Top (cmdb_id, kpi_name) pairs (examples from top 20): shippingservice2-0 / mesh.source.shippingservice2.jaeger-collector... (count 30), multiple adservice KPIs such as app.grpc.count, app.grpc.mrt, app.http.count, runtime.java_lang_* (each count 30). Many top pairs have count = 30.\n\n"
    "trace.csv:\n"
    "- Total rows: 36,092\n"
    "- Time range: 2022-03-20 21:00:00+08 to 2022-03-20 21:29:00+08\n"
    "- Unique cmdb_id: 40 (sample: adservice-0, adservice-1, adservice-2, adservice2-0, ...)\n"
    "- Unique trace_name count: 200\n"
    "- Top trace names by count (examples): trace.from_frontend-2.row_count (628), trace.from_frontend-2.error_rate (628), trace.from_frontend-2.duration_p95 (628), trace.from_frontend-2.duration_mean (628), trace.from_frontend-0.duration_mean (620), ...\n\n"
    "log.csv:\n"
    "- Total rows: 1,816\n"
    "- Time range: 2022-03-20 21:00:00+08 to 2022-03-20 21:29:00+08\n"
    "- Unique cmdb_id: 31 (sample: adservice-0, adservice-1, adservice-2, cartservice-...)\n"
    "- Unique log_name count: 2\n"
    "- Log name counts: log.error_count = 908, log.row_count = 908\n\n"
    "error_logs.csv:\n"
    "- Total rows: 1,592\n"
    "- Time range: 2022-03-20 21:00:00+08 to 2022-03-20 21:29:57+08\n"
    "- Unique cmdb_id: 3 (frontend-0, frontend-1, frontend-2)\n"
    "- Distinct messages: 1\n"
    "- Top message (repeated 1,592 times): \"severity: warning, message: failed to retrieve...\" (truncated in view)\n\n"
    "If you want more detail (e.g., exact top-50 KPI names, full top-20 (cmdb_id,kpi_name) table, or specific cmdb_id lists), ask and I will display the requested head slices."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(         file  total_rows       min_timestamp_UTC+8       max_timestamp_UTC+8  unique_cmdb_count  unique_kpi_count                 sample_unique_cmdb_list_(first_50)
0  metric.csv      145092 2022-03-20 21:00:00+08:00 2022-03-20 21:29:00+08:00                 59              1176  adservice,adservice-0,adservice-1,adservice-2,...,                                                 count  count
0   container.node-5.container_network_transmit_pa...   1050
1   container.node-5.container_network_transmit_er...   1050
2   container.node-5.container_network_receive_pac...   1050
3   container.node-5.container_network_receive_MB....   1050
4   container.node-5.container_network_receive_pac...   1050
..                                                ...    ...
45  container.node-5.container_memory_failures.con...    990
46  container.node-5.container_memory_failures.con...    990
47  container.node-5.container_memory_failures.hie...    990
48  container.node-5.container_fs_reads_merged./de...    990
49  container.node-5.container_fs_sector_reads./de...    990

[50 rows x 2 columns],                  cmdb_id                                           kpi_name  count
4845  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     30
0              adservice                                     app.grpc.count     30
1              adservice                                       app.grpc.mrt     30
2              adservice                                        app.grpc.rr     30
3              adservice                                        app.grpc.sr     30
4              adservice                                     app.http.count     30
5              adservice                                       app.http.mrt     30
6              adservice                                        app.http.rr     30
7              adservice                                        app.http.sr     30
8              adservice    runtime.java_lang_ClassLoading_LoadedClassCount     30
9              adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...     30
10             adservice  runtime.java_lang_ClassLoading_UnloadedClassCount     30
11             adservice             runtime.java_lang_ClassLoading_Verbose     30
12             adservice  runtime.java_lang_Compilation_CompilationTimeM...     30
4806  shippingservice2-0  container.node-5.container_network_receive_MB....     30
4807  shippingservice2-0  container.node-5.container_network_receive_err...     30
4808  shippingservice2-0  container.node-5.container_network_receive_pac...     30
4809  shippingservice2-0  container.node-5.container_network_receive_pac...     30
4810  shippingservice2-0  container.node-5.container_network_transmit_MB...     30
4811  shippingservice2-0  container.node-5.container_network_transmit_er...     30,         file  total_rows       min_timestamp_UTC+8       max_timestamp_UTC+8  unique_cmdb_count  unique_trace_name_count                 sample_unique_cmdb_list_(first_50)
0  trace.csv       36092 2022-03-20 21:00:00+08:00 2022-03-20 21:29:00+08:00                 40                      200  adservice-0,adservice-1,adservice-2,adservice2...,                                        count  count
0            trace.from_frontend-2.row_count    628
1           trace.from_frontend-2.error_rate    628
2         trace.from_frontend-2.duration_p95    628
3        trace.from_frontend-2.duration_mean    628
4        trace.from_frontend-0.duration_mean    620
..                                       ...    ...
45       trace.from_frontend2-0.duration_p95    205
46         trace.from_frontend2-0.error_rate    205
47          trace.from_frontend2-0.row_count    205
48  trace.to_currencyservice-2.duration_mean    180
49   trace.to_currencyservice-2.duration_p95    180

[50 rows x 2 columns],       file  total_rows       min_timestamp_UTC+8       max_timestamp_UTC+8  unique_cmdb_count  unique_log_name_count                 sample_unique_cmdb_list_(first_50)
0  log.csv        1816 2022-03-20 21:00:00+08:00 2022-03-20 21:29:00+08:00                 31                      2  adservice-0,adservice-1,adservice-2,cartservic...,              count  count
0  log.error_count    908
1    log.row_count    908,              file  total_rows       min_timestamp_UTC+8       max_timestamp_UTC+8  unique_cmdb_count  distinct_message_count sample_unique_cmdb_list_(first_50)
0  error_logs.csv        1592 2022-03-20 21:00:00+08:00 2022-03-20 21:29:57+08:00                  3                       1   frontend-0,frontend-1,frontend-2,                                                count  count
0  severity: warning, message: failed to retrieve...   1592)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

# Reuse existing df_metric and to_local_dt, tz from previous session
import pandas as pd
import pytz

# Define incident window in UTC+8
start_window = pd.Timestamp('2022-03-20 21:00:00', tz=tz)
end_window = pd.Timestamp('2022-03-20 21:30:00', tz=tz)  # exclusive

# 1) Compute global stats per (cmdb_id, kpi_name) using full series (before filtering)
g = df_metric.groupby(['cmdb_id', 'kpi_name'])['value']
pairs = (
    g.agg(
        total_points='count',
        global_p95=lambda x: x.quantile(0.95),
        global_p5=lambda x: x.quantile(0.05),
    )
    .reset_index()
)

# 2) Prepare metric rows with localized datetime and filter to incident window
# add localized datetime column (UTC+8)
if 'dt_local' not in df_metric.columns:
    df_metric['dt_local'] = to_local_dt(df_metric['timestamp'])

mask_window = (df_metric['dt_local'] >= start_window) & (df_metric['dt_local'] < end_window)
df_w = df_metric[mask_window].copy()

# If no rows in window, df_w will be empty; handle gracefully
# Merge global thresholds into window rows
df_w = df_w.merge(pairs, on=['cmdb_id', 'kpi_name'], how='left', validate='m:1')

# Mark anomalies in window using global thresholds
df_w['anom_above'] = df_w['value'] >= df_w['global_p95']
df_w['anom_below'] = df_w['value'] <= df_w['global_p5']
df_w['is_anom'] = df_w['anom_above'] | df_w['anom_below']

# 3) Aggregate per-pair stats within the window
if not df_w.empty:
    agg_window = df_w.groupby(['cmdb_id', 'kpi_name']).agg(
        points_in_window = ('value', 'size'),
        anomaly_count_above_p95 = ('anom_above', 'sum'),
        anomaly_count_below_p5 = ('anom_below', 'sum'),
        # earliest anomaly timestamp in window: use dt_local where is_anom True
        earliest_anomaly_ts_in_window = ('dt_local', lambda dt: dt[df_w.loc[dt.index, 'is_anom']].min() if dt[df_w.loc[dt.index, 'is_anom']].any() else pd.NaT),
        max_value_in_window = ('value', 'max'),
        min_value_in_window = ('value', 'min'),
    ).reset_index()
else:
    # empty structure consistent with columns
    agg_window = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','points_in_window','anomaly_count_above_p95',
        'anomaly_count_below_p5','earliest_anomaly_ts_in_window','max_value_in_window','min_value_in_window'
    ])

# 4) Combine global stats with window aggregates; only consider pairs that have points in window
combined = pairs.merge(agg_window, on=['cmdb_id','kpi_name'], how='inner', validate='1:1')

# Ensure numeric types
for col in ['global_p95','global_p5','max_value_in_window','min_value_in_window']:
    if col in combined:
        combined[col] = pd.to_numeric(combined[col], errors='coerce')

# Compute anomaly sum for sorting
combined['anomaly_sum'] = combined['anomaly_count_above_p95'].fillna(0).astype(int) + combined['anomaly_count_below_p5'].fillna(0).astype(int)

# 5) Decide whether anomalies exist; if none, rank by deviation (max_value_in_window - global_p95)
if combined['anomaly_sum'].sum() == 0:
    # compute deviation; if max_value_in_window missing (shouldn't for pairs in window), treat as -inf
    combined['deviation_over_p95'] = combined['max_value_in_window'] - combined['global_p95']
    combined_sorted = combined.sort_values(by='deviation_over_p95', ascending=False)
else:
    combined_sorted = combined.sort_values(by=['anomaly_sum','anomaly_count_above_p95','points_in_window'], ascending=[False, False, False])

# 6) Prepare final compact table with requested columns and formatting; limit to top 20
final_cols = [
    'cmdb_id','kpi_name','total_points','global_p95','global_p5',
    'points_in_window','anomaly_count_above_p95','anomaly_count_below_p5',
    'earliest_anomaly_ts_in_window','max_value_in_window','min_value_in_window'
]
# Ensure earliest_anomaly_ts_in_window displayed in UTC+8 ISO format (or NaT)
if 'earliest_anomaly_ts_in_window' in combined_sorted.columns:
    combined_sorted['earliest_anomaly_ts_in_window'] = combined_sorted['earliest_anomaly_ts_in_window'].dt.tz_convert(tz)

result_top20 = combined_sorted[final_cols].head(20)

# Display the compact result (top 20 rows)
result_top20

```
Out[2]:
```
summary = (
    "Metric.csv analysis summary (global thresholds computed on full series):\n\n"
    "- Incident window: 2022-03-20 21:00:00 to 2022-03-20 21:30:00 (UTC+8).\n\n"
    "- Top 20 (cmdb_id, kpi_name) by anomaly count are all from 'adservice'.\n"
    "  For each of these pairs:\n"
    "    • total_points (full series): 30\n"
    "    • points_in_window: 30\n"
    "    • global_p95 and global_p5 are equal to the observed values in the window\n"
    "      (examples: app.grpc.rr = 100.0, app.http.count = 24.0, runtime.java_lang_ClassLoading_LoadedClassCount = 4990.0, ...).\n"
    "    • anomaly_count_above_p95 = 30 and anomaly_count_below_p5 = 30\n"
    "    • earliest anomaly timestamp in window: 2022-03-20 21:00:00+08\n"
    "    • max_value_in_window and min_value_in_window are identical to the threshold value (no variation).\n\n"
    "Interpretation: For these top pairs the metric values are constant in the window and equal to both the 95th and 5th percentiles computed from their full series. As a result every point meets both >=global_p95 and <=global_p5 conditions. This likely indicates constant-valued series (or thresholds collapsing to the same value on small/identical data) rather than distinct high/low anomalies. If you want, I can (a) show the exact top-20 table rows again, or (b) expand analysis to other pairs or to non-adservice components."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  total_points  global_p95   global_p5  points_in_window  anomaly_count_above_p95  anomaly_count_below_p5 earliest_anomaly_ts_in_window  max_value_in_window  min_value_in_window
2   adservice                                        app.grpc.rr            30       100.0       100.0                30                       30                      30     2022-03-20 21:00:00+08:00                100.0                100.0
4   adservice                                     app.http.count            30        24.0        24.0                30                       30                      30     2022-03-20 21:00:00+08:00                 24.0                 24.0
6   adservice                                        app.http.rr            30       100.0       100.0                30                       30                      30     2022-03-20 21:00:00+08:00                100.0                100.0
7   adservice                                        app.http.sr            30       100.0       100.0                30                       30                      30     2022-03-20 21:00:00+08:00                100.0                100.0
8   adservice    runtime.java_lang_ClassLoading_LoadedClassCount            30      4990.0      4990.0                30                       30                      30     2022-03-20 21:00:00+08:00               4990.0               4990.0
9   adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...            30      4990.0      4990.0                30                       30                      30     2022-03-20 21:00:00+08:00               4990.0               4990.0
10  adservice  runtime.java_lang_ClassLoading_UnloadedClassCount            30         0.0         0.0                30                       30                      30     2022-03-20 21:00:00+08:00                  0.0                  0.0
11  adservice             runtime.java_lang_ClassLoading_Verbose            30         0.0         0.0                30                       30                      30     2022-03-20 21:00:00+08:00                  0.0                  0.0
12  adservice  runtime.java_lang_Compilation_CompilationTimeM...            30         1.0         1.0                30                       30                      30     2022-03-20 21:00:00+08:00                  1.0                  1.0
13  adservice  runtime.java_lang_Compilation_TotalCompilation...            30     43604.0     43604.0                30                       30                      30     2022-03-20 21:00:00+08:00              43604.0              43604.0
15  adservice  runtime.java_lang_GarbageCollector_CollectionC...            30         3.0         3.0                30                       30                      30     2022-03-20 21:00:00+08:00                  3.0                  3.0
17  adservice  runtime.java_lang_GarbageCollector_CollectionT...            30       177.0       177.0                30                       30                      30     2022-03-20 21:00:00+08:00                177.0                177.0
18  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...            30         1.0         1.0                30                       30                      30     2022-03-20 21:00:00+08:00                  1.0                  1.0
19  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...            30         1.0         1.0                30                       30                      30     2022-03-20 21:00:00+08:00                  1.0                  1.0
21  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...            30        63.0        63.0                30                       30                      30     2022-03-20 21:00:00+08:00                 63.0                 63.0
23  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...            30     56457.0     56457.0                30                       30                      30     2022-03-20 21:00:00+08:00              56457.0              56457.0
25  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...            30         3.0         3.0                30                       30                      30     2022-03-20 21:00:00+08:00                  3.0                  3.0
26  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...            30  22937600.0  22937600.0                30                       30                      30     2022-03-20 21:00:00+08:00           22937600.0           22937600.0
27  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...            30   7340032.0   7340032.0                30                       30                      30     2022-03-20 21:00:00+08:00            7340032.0            7340032.0
28  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...            30   3801088.0   3801088.0                30                       30                      30     2022-03-20 21:00:00+08:00            3801088.0            3801088.0```
```

In [ ]:
# In[3]:

# Reuse existing variables: df_metric, to_local_dt, tz
import pandas as pd
import numpy as np
import pytz

# Ensure dt_local exists
if 'dt_local' not in df_metric.columns:
    df_metric['dt_local'] = to_local_dt(df_metric['timestamp'])

# Define incident window in UTC+8
start_window = pd.Timestamp('2022-03-20 21:00:00', tz=tz)
end_window = pd.Timestamp('2022-03-20 21:30:00', tz=tz)  # exclusive

# 1) Compute global stats per (cmdb_id, kpi_name) using the full series (thresholds BEFORE filtering)
pairs = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(
        total_points='count',
        global_p95=lambda s: s.quantile(0.95),
        global_p5=lambda s: s.quantile(0.05),
    )
    .reset_index()
)

# 2) Filter metric rows to incident window
mask_window = (df_metric['dt_local'] >= start_window) & (df_metric['dt_local'] < end_window)
df_w = df_metric[mask_window].copy()

# If no rows in window, produce empty outputs gracefully
if df_w.shape[0] == 0:
    result_top20 = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','total_points','global_p95','global_p5',
        'points_in_window','anomaly_count_above_p95','anomaly_count_below_p5',
        'earliest_anomaly_ts_in_window','max_value_in_window','min_value_in_window',
        'std_in_window','deviation_score'
    ])
    distinct_cmdbids = pd.Series(dtype=object)
else:
    # 3) Merge global thresholds into window rows (global thresholds computed above)
    df_w = df_w.merge(pairs, on=['cmdb_id','kpi_name'], how='left', validate='m:1')

    # 4) Mark anomalies in window using global thresholds
    df_w['anom_above'] = df_w['value'] >= df_w['global_p95']
    df_w['anom_below'] = df_w['value'] <= df_w['global_p5']
    df_w['is_anom'] = df_w['anom_above'] | df_w['anom_below']

    # 5) Aggregate per-pair stats within the window
    grp = df_w.groupby(['cmdb_id','kpi_name'])
    window_stats = grp.agg(
        points_in_window = ('value', 'size'),
        anomaly_count_above_p95 = ('anom_above', 'sum'),
        anomaly_count_below_p5 = ('anom_below', 'sum'),
        max_value_in_window = ('value', 'max'),
        min_value_in_window = ('value', 'min'),
        std_in_window = ('value', 'std')  # sample std (ddof=1)
    ).reset_index()

    # earliest anomaly timestamp per pair (where is_anom True)
    df_w_anom = df_w[df_w['is_anom']].groupby(['cmdb_id','kpi_name'])['dt_local'].min().reset_index().rename(columns={'dt_local':'earliest_anomaly_ts_in_window'})

    # Merge earliest anomaly
    window_stats = window_stats.merge(df_w_anom, on=['cmdb_id','kpi_name'], how='left', validate='1:1')

    # 6) Combine with global thresholds (only pairs that have points in window)
    combined = pairs.merge(window_stats, on=['cmdb_id','kpi_name'], how='inner', validate='1:1')

    # 7) Flag constant series and compute deviation_score
    combined['constant_series'] = combined['max_value_in_window'] == combined['min_value_in_window']
    # compute positive deltas, else 0
    delta_above = (combined['max_value_in_window'] - combined['global_p95']).clip(lower=0)
    delta_below = (combined['global_p5'] - combined['min_value_in_window']).clip(lower=0)
    combined['deviation_score'] = np.maximum(delta_above.fillna(0), delta_below.fillna(0))

    # 8) Exclude constant series True
    combined_nonconst = combined[~combined['constant_series']].copy()

    # If after exclusion no pairs remain, fall back to original combined (but instruction says exclude; still handle)
    if combined_nonconst.shape[0] == 0:
        # produce empty result as spec requests excluding constant_series True
        result_top20 = pd.DataFrame(columns=[
            'cmdb_id','kpi_name','total_points','global_p95','global_p5',
            'points_in_window','anomaly_count_above_p95','anomaly_count_below_p5',
            'earliest_anomaly_ts_in_window','max_value_in_window','min_value_in_window',
            'std_in_window','deviation_score'
        ])
        distinct_cmdbids = pd.Series(dtype=object)
    else:
        # 9) Sort by deviation_score descending and take top20
        combined_nonconst_sorted = combined_nonconst.sort_values(by='deviation_score', ascending=False)

        # 10) Prepare final columns and format earliest timestamp
        final_cols = [
            'cmdb_id','kpi_name','total_points','global_p95','global_p5',
            'points_in_window','anomaly_count_above_p95','anomaly_count_below_p5',
            'earliest_anomaly_ts_in_window','max_value_in_window','min_value_in_window',
            'std_in_window','deviation_score'
        ]
        # Ensure earliest timestamp is timezone-aware in UTC+8 (dt_local already is)
        if 'earliest_anomaly_ts_in_window' in combined_nonconst_sorted.columns:
            # Some earliest timestamps may be NaT; keep as is
            # They are already in tz=Asia/Shanghai
            pass

        result_top20 = combined_nonconst_sorted[final_cols].head(20).reset_index(drop=True)

        # Distinct cmdb_id among these top20 (compact)
        distinct_cmdbids = pd.Series(result_top20['cmdb_id'].unique()).sort_values().reset_index(drop=True)

# Display result (DataFrame of top20, and compact list of distinct cmdb_id)
result_top20, distinct_cmdbids

```
Out[3]:
```
Summary (incident window 2022-03-20 21:00–21:30 UTC+8).  
I excluded constant-valued series and ranked non-constant (cmdb_id, kpi_name) pairs by deviation_score (largest positive deviation from global thresholds). Key findings from the top 20:

Top candidates (compact highlights)
- adservice2 / runtime.java_lang_OperatingSystem_ProcessCpuTime
  - total_points: 30, points_in_window: 30
  - global_p95: 8.4808e+10, global_p5: 7.9509e+10
  - anomaly_count_above_p95: 2, anomaly_count_below_p5: 2
  - earliest anomaly: 2022-03-20 21:00:00+08
  - max/min in window: 8.50625e+10 / 7.895833e+10
  - std_in_window: 1.7626e+09, deviation_score: 5.5083e+08

- adservice / runtime.java_lang_OperatingSystem_ProcessCpuTime
  - total_points: 30, points_in_window: 30
  - global_p95: 8.4916e+11, global_p5: 8.4434e+11
  - anomalies above/below: 2 / 2
  - earliest anomaly: 2022-03-20 21:00:00+08
  - max/min in window: 8.49415e+11 / 8.440633e+11
  - std_in_window: 1.6265e+09, deviation_score: 2.7458e+08

- adservice2 / runtime.java_lang_Threading_CurrentThreadCpuTime
  - (30 points; 2 above / 2 below; deviation_score ≈ 5.78e+07)

- adservice and adservice2 several threading/alloc/runtime JVM metrics
  - generally: 30 points in window, small numbers of points above/below global thresholds (typically 2 above and 2 below), non-zero std, deviation_scores ranging from ~2e7 to ~5e8.

Other notable entries
- container.memory metrics on node-6 for adservice2-0 and frontend2-0:
  - e.g., adservice2-0 / container.node-6.container_memory_cache: anomaly_count_above_p95 = 2, anomaly_count_below_p5 = 9, std_in_window ~8.15e6, deviation_score ~4.96e6.
- node-level filesystem / disk free & inodes metrics (node-3, node-4, node-6) appear in top list:
  - all have 30 points in window, 2 above / 2 below, non-zero std, deviation_scores in millions to hundreds of millions.
- currencyservice2-0 container memory rss and failcnt also appear with small anomaly counts (2/2) and positive deviation_score.

Distinct cmdb_id values among the top 20 (sorted)
- adservice, adservice2, adservice2-0, currencyservice2-0, frontend2-0, node-3, node-4, node-6

Interpretation and notes
- All top pairs have full-series thresholds computed on all available data; the episode window contains 30 samples per pair (one per minute).
- Most top pairs show only a few anomalies relative to global thresholds (commonly 2 points above and 2 below). Deviation_score is the larger of (max_in_window - global_p95) and (global_p5 - min_in_window) and indicates how far the window extremes move past global thresholds.
- Some container memory metrics showed more below-p5 counts (e.g., cache metrics with 9 below_p5), which may indicate resource variation in the window worth investigating.
- Earliest anomaly for all reported pairs is 2022-03-20 21:00:00+08 (the start of the window).

Recommended next steps
- Inspect the JVM CPU/threading and container memory metrics for the listed adservice/adservice2 pods (and the node-level filesystem/disk metrics) around and before 21:00 to see trending or correlated events.
- For container memory_cache metrics with higher below-p5 counts, check container memory pressure or eviction logs for those pods/nodes.
- Correlate these metric hits with logs/traces for the same cmdb_ids to confirm root cause.

If you want, I can: (a) show the full top-20 table rows verbatim, (b) expand to include constant series, or (c) cross-correlate these top cmdb_ids with trace/log/error data. Which would you prefer?

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id                                           kpi_name  total_points    global_p95     global_p5  points_in_window  anomaly_count_above_p95  anomaly_count_below_p5 earliest_anomaly_ts_in_window  max_value_in_window  min_value_in_window  std_in_window  deviation_score
0           adservice2   runtime.java_lang_OperatingSystem_ProcessCpuTime            30  8.480792e+10  7.950917e+10                30                        2                       2     2022-03-20 21:00:00+08:00         8.506250e+10         7.895833e+10   1.762638e+09     5.508333e+08
1            adservice   runtime.java_lang_OperatingSystem_ProcessCpuTime            30  8.491573e+11  8.443379e+11                30                        2                       2     2022-03-20 21:00:00+08:00         8.494150e+11         8.440633e+11   1.626525e+09     2.745833e+08
2           adservice2   runtime.java_lang_Threading_CurrentThreadCpuTime            30  5.220553e+09  4.704228e+09                30                        2                       2     2022-03-20 21:00:00+08:00         5.278308e+09         4.662268e+09   1.799978e+08     5.775497e+07
3           adservice2  runtime.java_lang_Threading_CurrentThreadUserTime            30  4.862708e+09  4.379917e+09                30                        2                       2     2022-03-20 21:00:00+08:00         4.918333e+09         4.341667e+09   1.679923e+08     5.562500e+07
4            adservice  runtime.java_lang_Threading_CurrentThreadUserTime            30  7.778550e+10  7.732062e+10                30                        2                       2     2022-03-20 21:00:00+08:00         7.783750e+10         7.727083e+10   1.595613e+08     5.200000e+07
5            adservice   runtime.java_lang_Threading_CurrentThreadCpuTime            30  7.997039e+10  7.948869e+10                30                        2                       2     2022-03-20 21:00:00+08:00         8.001580e+10         7.944554e+10   1.640699e+08     4.540607e+07
6           adservice2  runtime.java_lang_Threading_CurrentThreadAlloc...            30  3.600193e+09  3.216659e+09                30                        2                       2     2022-03-20 21:00:00+08:00         3.622891e+09         3.194709e+09   1.296188e+08     2.269811e+07
7            adservice  runtime.java_lang_Threading_CurrentThreadAlloc...            30  6.334743e+10  6.296547e+10                30                        2                       2     2022-03-20 21:00:00+08:00         6.336902e+10         6.294406e+10   1.288305e+08     2.159417e+07
8         adservice2-0            container.node-6.container_memory_cache            30  5.862502e+07  4.128358e+07                30                        2                       9     2022-03-20 21:00:00+08:00         6.358630e+07         4.128358e+07   8.146914e+06     4.961280e+06
9           adservice2  runtime.java_lang_OperatingSystem_FreePhysical...            30  9.911910e+07  8.248934e+07                30                        4                       3     2022-03-20 21:00:00+08:00         9.911910e+07         7.866914e+07   7.814729e+06     3.820203e+06
10  currencyservice2-0              container.node-6.container_memory_rss            30  5.445612e+07  5.305167e+07                30                        2                       2     2022-03-20 21:00:00+08:00         5.465702e+07         5.052211e+07   7.230471e+05     2.529553e+06
11              node-4                              system.fs.inodes.free            30  6.625745e+09  6.615459e+09                30                        2                       2     2022-03-20 21:00:00+08:00         6.628241e+09         6.615194e+09   3.361101e+06     2.496410e+06
12              node-4                             system.fs.inodes.total            30  6.628332e+09  6.618046e+09                30                        2                       2     2022-03-20 21:00:00+08:00         6.630827e+09         6.617781e+09   3.360906e+06     2.495727e+06
13              node-6                                   system.disk.free            30  1.364630e+09  1.221048e+09                30                        2                       2     2022-03-20 21:00:00+08:00         1.365142e+09         1.218826e+09   4.304352e+07     2.221491e+06
14              node-3                              system.fs.inodes.free            30  4.978653e+09  4.971278e+09                30                        2                       2     2022-03-20 21:00:00+08:00         4.980836e+09         4.971055e+09   2.431049e+06     2.183014e+06
15              node-3                             system.fs.inodes.total            30  4.980600e+09  4.973226e+09                30                        2                       2     2022-03-20 21:00:00+08:00         4.982783e+09         4.973002e+09   2.431022e+06     2.182733e+06
16         frontend2-0            container.node-6.container_memory_cache            30  3.071621e+07  2.588672e+07                30                        2                       9     2022-03-20 21:00:00+08:00         3.278848e+07         2.588672e+07   2.265848e+06     2.072269e+06
17              node-6                             system.fs.inodes.total            30  4.431457e+09  3.739721e+09                30                        2                       2     2022-03-20 21:00:00+08:00         4.433427e+09         3.739576e+09   2.106304e+08     1.969311e+06
18              node-6                              system.fs.inodes.free            30  4.429548e+09  3.738034e+09                30                        2                       2     2022-03-20 21:00:00+08:00         4.431517e+09         3.737890e+09   2.105623e+08     1.969311e+06
19  currencyservice2-0          container.node-6.container_memory_failcnt            30  7.994509e+08  7.667961e+08                30                        2                       2     2022-03-20 21:00:00+08:00         8.012790e+08         7.648835e+08   1.093311e+07     1.912632e+06, 0             adservice
1            adservice2
2          adservice2-0
3    currencyservice2-0
4           frontend2-0
5                node-3
6                node-4
7                node-6
dtype: object)```
```

In [ ]:
# In[4]:

# Reuse existing df_trace, to_local_dt, tz if available; else import/define
import pandas as pd
import numpy as np
import pytz

# Ensure timezone variable
tz = pytz.timezone('Asia/Shanghai')

# Candidate cmdb_id set
candidates = ['adservice','adservice2','adservice2-0','currencyservice2-0','frontend2-0','node-3','node-4','node-6']

# Ensure dt_local column on df_trace
if 'dt_local' not in df_trace.columns:
    df_trace['dt_local'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Restrict to candidate cmdb_ids (full series for those IDs)
df_t = df_trace[df_trace['cmdb_id'].isin(candidates)].copy()

# If empty, produce empty outputs
if df_t.shape[0] == 0:
    result_top20 = pd.DataFrame(columns=[
        'cmdb_id','trace_name','total_points','global_p95','global_p5',
        'points_in_window','anomaly_count_above_p95','anomaly_count_below_p5',
        'earliest_anomaly_ts_in_window','avg_value_in_window','max_value_in_window','min_value_in_window'
    ])
    trace_type_list = pd.Series(dtype=object)
else:
    # 1) Compute global thresholds per (cmdb_id, trace_name) using full series (before window filtering)
    pairs = (
        df_t
        .groupby(['cmdb_id','trace_name'])['value']
        .agg(
            total_points = 'count',
            global_p95 = lambda s: s.quantile(0.95),
            global_p5  = lambda s: s.quantile(0.05)
        )
        .reset_index()
    )

    # 2) Filter to incident window (UTC+8)
    start_window = pd.Timestamp('2022-03-20 21:00:00', tz=tz)
    end_window = pd.Timestamp('2022-03-20 21:30:00', tz=tz)  # exclusive
    mask_window = (df_t['dt_local'] >= start_window) & (df_t['dt_local'] < end_window)
    df_w = df_t[mask_window].copy()

    if df_w.shape[0] == 0:
        # No window rows
        result_top20 = pd.DataFrame(columns=[
            'cmdb_id','trace_name','total_points','global_p95','global_p5',
            'points_in_window','anomaly_count_above_p95','anomaly_count_below_p5',
            'earliest_anomaly_ts_in_window','avg_value_in_window','max_value_in_window','min_value_in_window'
        ])
        trace_type_list = pd.Series(dtype=object)
    else:
        # 3) Merge global thresholds into window rows
        df_w = df_w.merge(pairs, on=['cmdb_id','trace_name'], how='left', validate='m:1')

        # 4) Mark anomalies
        df_w['anom_above'] = df_w['value'] >= df_w['global_p95']
        df_w['anom_below'] = df_w['value'] <= df_w['global_p5']
        df_w['is_anom'] = df_w['anom_above'] | df_w['anom_below']

        # 5) Aggregate window stats per pair
        grp = df_w.groupby(['cmdb_id','trace_name'])
        window_stats = grp.agg(
            points_in_window = ('value','size'),
            anomaly_count_above_p95 = ('anom_above','sum'),
            anomaly_count_below_p5 = ('anom_below','sum'),
            avg_value_in_window = ('value','mean'),
            max_value_in_window = ('value','max'),
            min_value_in_window = ('value','min')
        ).reset_index()

        # earliest anomaly timestamp where either condition holds
        df_w_anom = df_w[df_w['is_anom']].groupby(['cmdb_id','trace_name'])['dt_local'].min().reset_index().rename(columns={'dt_local':'earliest_anomaly_ts_in_window'})

        window_stats = window_stats.merge(df_w_anom, on=['cmdb_id','trace_name'], how='left', validate='1:1')

        # 6) Combine with global thresholds (only pairs that have points in window)
        combined = pairs.merge(window_stats, on=['cmdb_id','trace_name'], how='inner', validate='1:1')

        # Fill NaNs for counts with 0 (shouldn't be necessary) and ensure numeric types
        combined['anomaly_count_above_p95'] = combined['anomaly_count_above_p95'].fillna(0).astype(int)
        combined['anomaly_count_below_p5'] = combined['anomaly_count_below_p5'].fillna(0).astype(int)
        combined['points_in_window'] = combined['points_in_window'].fillna(0).astype(int)

        # 7) Sort by anomaly sum desc, tie-break earliest anomaly asc (NaT will go last)
        combined['anomaly_sum'] = combined['anomaly_count_above_p95'] + combined['anomaly_count_below_p5']
        combined_sorted = combined.sort_values(by=['anomaly_sum','earliest_anomaly_ts_in_window'], ascending=[False, True])

        # 8) Select requested columns and top 20
        final_cols = [
            'cmdb_id','trace_name','total_points','global_p95','global_p5',
            'points_in_window','anomaly_count_above_p95','anomaly_count_below_p5',
            'earliest_anomaly_ts_in_window','avg_value_in_window','max_value_in_window','min_value_in_window'
        ]
        result_top20 = combined_sorted[final_cols].head(20).reset_index(drop=True)

        # 9) Short list of trace_name types among results that indicate error_rate or duration or row_count
        trace_names_in_results = result_top20['trace_name'].astype(str).unique()
        types = []
        for t in trace_names_in_results:
            if 'error_rate' in t and 'error_rate' not in types:
                types.append('error_rate')
            if 'duration' in t and 'duration' not in types:
                types.append('duration')
            if 'row_count' in t and 'row_count' not in types:
                types.append('row_count')
        trace_type_list = pd.Series(types)

# Display compact outputs: result_top20 and trace_type_list
result_top20, trace_type_list

```
Out[4]:
```
summary = (
    "Trace.csv analysis for candidate cmdb_ids [adservice, adservice2, adservice2-0, currencyservice2-0, "
    "frontend2-0, node-3, node-4, node-6] (incident window 2022-03-20 21:00–21:30 UTC+8):\n\n"
    "Key findings:\n"
    "- Many trace series are constant-zero error_rate metrics (e.g. trace.*.error_rate). For these, global_p95 and global_p5 = 0 and every point in the window (30 samples) equals 0, so they register as both >=p95 and <=p5. Examples include: adservice2-0.trace.from_frontend2-0.error_rate, currencyservice2-0.trace.from_frontend2-0.error_rate, frontend2-0.trace.from_root.error_rate, and several trace.to_* error_rate series on frontend2-0. Earliest anomaly times are 2022-03-20 21:00:00+08.\n\n"
    "- Non-constant traces with meaningful variation include row_count and duration metrics:\n"
    "  • frontend2-0.trace.to_checkoutservice2-0.row_count (25 points in window): avg=2.12, max=4, min=1, anomaly counts above/below p95/p5 = 6 / 11, earliest anomaly 21:00:00+08.\n"
    "  • currencyservice2-0.trace.from_checkoutservice2-0.row_count (25 points): avg=6.12, max=12, min=2, anomaly counts = 5 / 2, earliest anomaly 21:02:00+08.\n"
    "  • Several duration metrics (duration_p95, duration_mean) on adservice2-0 and frontend2-0 show small but measurable deviations with earliest anomalies around 21:01–21:03+08 (e.g. trace.from_frontend2-0.duration_p95 and trace.to_adservice2-0.duration_p95).\n\n"
    "Aggregate behavior and timestamps:\n"
    "- Most reported pairs have full-series total_points of 30 (some row_count traces have 25). Many earliest anomalies occur at the window start (21:00:00+08); a few duration/row_count anomalies appear slightly later (21:01–21:03+08).\n"
    "- For error_rate traces that are all-zero, the anomaly counts are artifacts of thresholds collapsing to zero; they do not indicate spikes but constant-zero series.\n\n"
    "Trace types present among top results: error_rate, duration, row_count.\n\n"
    "Recommendations:\n"
    "1) Ignore constant zero error_rate series for anomaly-significance (they reflect no errors rather than anomalies). Focus on non-constant row_count and duration traces.\n"
    "2) Investigate frontend2-0 and adservice2-0 duration and row_count traces (to/from checkout/cart/adservice2) — they show the largest non-constant anomaly counts and earliest anomalies in the window.\n"
    "3) Correlate these trace hits with the metric-level candidates (adservice/adservice2/container memory and JVM CPU/thread metrics) and with logs for frontend2-0 and checkout-related services to narrow root cause.\n\n"
    "If you want, I can (a) show the top-20 trace rows verbatim, (b) filter out constant-zero series and re-rank, or (c) cross-correlate these trace timestamps with metric/log events."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id                                    trace_name  total_points  global_p95  global_p5  points_in_window  anomaly_count_above_p95  anomaly_count_below_p5 earliest_anomaly_ts_in_window  avg_value_in_window  max_value_in_window  min_value_in_window
0         adservice2-0             trace.from_frontend2-0.error_rate            30    0.000000   0.000000                30                       30                      30     2022-03-20 21:00:00+08:00             0.000000             0.000000             0.000000
1   currencyservice2-0             trace.from_frontend2-0.error_rate            30    0.000000   0.000000                30                       30                      30     2022-03-20 21:00:00+08:00             0.000000             0.000000             0.000000
2          frontend2-0                    trace.from_root.error_rate            30    0.000000   0.000000                30                       30                      30     2022-03-20 21:00:00+08:00             0.000000             0.000000             0.000000
3          frontend2-0                         trace.self.error_rate            30    0.000000   0.000000                30                       30                      30     2022-03-20 21:00:00+08:00             0.000000             0.000000             0.000000
4          frontend2-0              trace.to_adservice2-0.error_rate            30    0.000000   0.000000                30                       30                      30     2022-03-20 21:00:00+08:00             0.000000             0.000000             0.000000
5          frontend2-0          trace.to_cartservice2-0.duration_p95            30    0.001000   0.001000                30                       30                      30     2022-03-20 21:00:00+08:00             0.001000             0.001000             0.001000
6          frontend2-0            trace.to_cartservice2-0.error_rate            30    0.000000   0.000000                30                       30                      30     2022-03-20 21:00:00+08:00             0.000000             0.000000             0.000000
7          frontend2-0        trace.to_currencyservice2-0.error_rate            30    0.000000   0.000000                30                       30                      30     2022-03-20 21:00:00+08:00             0.000000             0.000000             0.000000
8          frontend2-0  trace.to_productcatalogservice2-0.error_rate            30    0.000000   0.000000                30                       30                      30     2022-03-20 21:00:00+08:00             0.000000             0.000000             0.000000
9          frontend2-0  trace.to_recommendationservice2-0.error_rate            30    0.000000   0.000000                30                       30                      30     2022-03-20 21:00:00+08:00             0.000000             0.000000             0.000000
10         frontend2-0        trace.to_shippingservice2-0.error_rate            30    0.000000   0.000000                30                       30                      30     2022-03-20 21:00:00+08:00             0.000000             0.000000             0.000000
11  currencyservice2-0      trace.from_checkoutservice2-0.error_rate            25    0.000000   0.000000                25                       25                      25     2022-03-20 21:00:00+08:00             0.000000             0.000000             0.000000
12         frontend2-0        trace.to_checkoutservice2-0.error_rate            25    0.000000   0.000000                25                       25                      25     2022-03-20 21:00:00+08:00             0.000000             0.000000             0.000000
13         frontend2-0         trace.to_checkoutservice2-0.row_count            25    4.000000   1.000000                25                        6                      11     2022-03-20 21:00:00+08:00             2.120000             4.000000             1.000000
14        adservice2-0           trace.from_frontend2-0.duration_p95            30    0.000030   0.000022                30                        3                       6     2022-03-20 21:01:00+08:00             0.000025             0.000032             0.000021
15         frontend2-0            trace.to_adservice2-0.duration_p95            30    0.000030   0.000022                30                        3                       6     2022-03-20 21:01:00+08:00             0.000025             0.000032             0.000021
16  currencyservice2-0       trace.from_checkoutservice2-0.row_count            25   12.000000   2.200000                25                        5                       2     2022-03-20 21:02:00+08:00             6.120000            12.000000             2.000000
17         frontend2-0     trace.to_shippingservice2-0.duration_mean            30    0.000086   0.000059                30                        2                       5     2022-03-20 21:03:00+08:00             0.000066             0.000116             0.000058
18        adservice2-0          trace.from_frontend2-0.duration_mean            30    0.000022   0.000018                30                        2                       4     2022-03-20 21:02:00+08:00             0.000020             0.000022             0.000018
19         frontend2-0           trace.to_adservice2-0.duration_mean            30    0.000022   0.000018                30                        2                       4     2022-03-20 21:02:00+08:00             0.000020             0.000022             0.000018, 0    error_rate
1      duration
2     row_count
dtype: object)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# Ensure timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Ensure dt_local exists for df_error and df_log (reuse if present)
def ensure_dt_local(df):
    if 'dt_local' not in df.columns:
        df['dt_local'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    return df

df_error = ensure_dt_local(df_error)
df_log = ensure_dt_local(df_log)

# Define incident window (UTC+8)
start_window = pd.Timestamp('2022-03-20 21:00:00', tz=tz)
end_window = pd.Timestamp('2022-03-20 21:30:00', tz=tz)  # exclusive

# -------------------------
# 1) error_logs.csv analysis
# -------------------------
# Filter error logs to window
mask_err_win = (df_error['dt_local'] >= start_window) & (df_error['dt_local'] < end_window)
df_error_win = df_error[mask_err_win].copy()

# Overall earliest timestamp across all cmdb_ids in window and total count
if not df_error_win.empty:
    overall_earliest_error_ts = df_error_win['dt_local'].min()
    total_error_logs_in_window = int(df_error_win.shape[0])
else:
    overall_earliest_error_ts = pd.NaT
    total_error_logs_in_window = 0

# Aggregate by cmdb_id: total_count, earliest and latest timestamp in window, sample_message_text (first 200 chars)
if not df_error_win.empty:
    # sort by dt_local so first() yields earliest message per cmdb_id
    df_error_win_sorted = df_error_win.sort_values(['cmdb_id','dt_local'])
    error_agg = (
        df_error_win_sorted
        .groupby('cmdb_id', sort=True)
        .agg(
            total_count = ('message', 'size'),
            earliest_timestamp_in_window = ('dt_local', 'min'),
            latest_timestamp_in_window = ('dt_local', 'max'),
            sample_message_text = ('message', lambda s: str(s.iloc[0])[:200])
        )
        .reset_index()
        .sort_values('total_count', ascending=False)
        .head(20)
    )
else:
    error_agg = pd.DataFrame(columns=['cmdb_id','total_count','earliest_timestamp_in_window','latest_timestamp_in_window','sample_message_text'])

# -------------------------
# 2) log.csv analysis
# -------------------------
# total_rows_for_cmdb (in full file)
total_rows_per_cmdb = df_log.groupby('cmdb_id').size().rename('total_rows_for_cmdb').reset_index()

# Filter log rows to window
mask_log_win = (df_log['dt_local'] >= start_window) & (df_log['dt_local'] < end_window)
df_log_win = df_log[mask_log_win].copy()

# points_in_window per cmdb_id
points_in_window = df_log_win.groupby('cmdb_id').size().rename('points_in_window').reset_index()

# For log.error_count within window: sum and max per cmdb_id
log_error_win = df_log_win[df_log_win['log_name']=='log.error_count'].copy()
if not log_error_win.empty:
    sum_err = log_error_win.groupby('cmdb_id')['value'].sum().rename('sum_log_error_count_in_window').reset_index()
    max_err = log_error_win.groupby('cmdb_id')['value'].max().rename('max_log_error_count_in_window').reset_index()
    # earliest timestamp in window where log.error_count > 0
    positive_err = log_error_win[log_error_win['value']>0]
    if not positive_err.empty:
        earliest_pos_err = positive_err.sort_values('dt_local').groupby('cmdb_id')['dt_local'].min().rename('earliest_timestamp_in_window_where_log.error_count>0').reset_index()
    else:
        earliest_pos_err = pd.DataFrame(columns=['cmdb_id','earliest_timestamp_in_window_where_log.error_count>0'])
else:
    sum_err = pd.DataFrame(columns=['cmdb_id','sum_log_error_count_in_window'])
    max_err = pd.DataFrame(columns=['cmdb_id','max_log_error_count_in_window'])
    earliest_pos_err = pd.DataFrame(columns=['cmdb_id','earliest_timestamp_in_window_where_log.error_count>0'])

# Merge per-cmdb results
log_summary = total_rows_per_cmdb.merge(points_in_window, on='cmdb_id', how='left')
log_summary = log_summary.merge(sum_err, on='cmdb_id', how='left')
log_summary = log_summary.merge(max_err, on='cmdb_id', how='left')
log_summary = log_summary.merge(earliest_pos_err, on='cmdb_id', how='left')

# Fill NaNs for numeric sums/max with 0
log_summary['points_in_window'] = log_summary['points_in_window'].fillna(0).astype(int)
log_summary['sum_log_error_count_in_window'] = log_summary['sum_log_error_count_in_window'].fillna(0)
log_summary['max_log_error_count_in_window'] = log_summary['max_log_error_count_in_window'].fillna(0)

# Unique log_name values per cmdb_id (from full file) — keep as comma-joined short list
log_names_per_cmdb = df_log.groupby('cmdb_id')['log_name'].unique().reset_index()
log_names_per_cmdb['unique_log_names'] = log_names_per_cmdb['log_name'].apply(lambda arr: ','.join(map(str, arr)) )
log_names_per_cmdb = log_names_per_cmdb[['cmdb_id','unique_log_names']]

log_summary = log_summary.merge(log_names_per_cmdb, on='cmdb_id', how='left')

# Select top 20 cmdb_ids by sum_log_error_count_in_window descending
log_summary_top20 = log_summary.sort_values('sum_log_error_count_in_window', ascending=False).head(20)

# Keep columns compact and ordered as requested
log_summary_top20 = log_summary_top20[[
    'cmdb_id','total_rows_for_cmdb','points_in_window',
    'sum_log_error_count_in_window','max_log_error_count_in_window',
    'earliest_timestamp_in_window_where_log.error_count>0','unique_log_names'
]]

# Final compact outputs
error_agg, overall_earliest_error_ts, total_error_logs_in_window, log_summary_top20

```
Out[5]:
```
summary = (
    "Incident window: 2022-03-20 21:00:00 to 2022-03-20 21:30:00 (UTC+8)\n\n"
    "1) error_logs.csv (aggregated by cmdb_id, top entries):\n"
    "- frontend-2: total_count=605, earliest=2022-03-20 21:00:00+08:00, latest=2022-03-20 21:29:57+08:00, sample_message=\"severity: warning, message: failed to retrieve...\"\n"
    "- frontend-0: total_count=548, earliest=2022-03-20 21:00:00+08:00, latest=2022-03-20 21:29:57+08:00, sample_message=\"severity: warning, message: failed to retrieve...\"\n"
    "- frontend-1: total_count=439, earliest=2022-03-20 21:00:01+08:00, latest=2022-03-20 21:29:57+08:00, sample_message=\"severity: warning, message: failed to retrieve...\"\n\n"
    "Overall: earliest error_log in window = 2022-03-20 21:00:00+08:00; total error_logs in window = 1,592.\n\n"
    "2) log.csv (top cmdb_ids by sum(log.error_count) in window):\n"
    "- frontend-2: total_rows_in_file=60, points_in_window=60, sum_log.error_count=605.0, max=27.0, earliest timestamp with error_count>0 = 2022-03-20 21:00:00+08:00, unique_log_names=log.error_count,log.row_count\n"
    "- frontend-0: total_rows_in_file=60, points_in_window=60, sum_log.error_count=548.0, max=24.0, earliest >0 = 2022-03-20 21:00:00+08:00, unique_log_names=log.error_count,log.row_count\n"
    "- frontend-1: total_rows_in_file=60, points_in_window=60, sum_log.error_count=439.0, max=19.0, earliest >0 = 2022-03-20 21:00:00+08:00, unique_log_names=log.error_count,log.row_count\n\n"
    "Many other cmdb_ids (e.g., adservice-0/1/2, cartservice-*, checkoutservice-*, currencyservice-*) show 60 rows in-window but sum_log.error_count = 0 and unique_log_names = log.error_count,log.row_count.\n\n"
    "Interpretation:\n"
    "- Frontend pods (frontend-2, frontend-0, frontend-1) are the dominant sources of both error_logs and elevated log.error_count during the window, with earliest hits at the window start (21:00).\n"
    "- Error messages are consistently the same warning text (sample truncated), repeated heavily on frontend pods.\n"
    "- Most other services recorded log rows but no log.error_count in this window.\n\n"
    "Recommended next steps: prioritize investigating frontend pods (logs and traces around 21:00), correlate with the metric and trace anomalies previously identified for frontend2-0 and adservice/adservice2."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id  total_count earliest_timestamp_in_window latest_timestamp_in_window                                sample_message_text
2  frontend-2          605    2022-03-20 21:00:00+08:00  2022-03-20 21:29:57+08:00  severity: warning, message: failed to retrieve...
0  frontend-0          548    2022-03-20 21:00:00+08:00  2022-03-20 21:29:57+08:00  severity: warning, message: failed to retrieve...
1  frontend-1          439    2022-03-20 21:00:01+08:00  2022-03-20 21:29:57+08:00  severity: warning, message: failed to retrieve..., Timestamp('2022-03-20 21:00:00+0800', tz='Asia/Shanghai'), 1592,               cmdb_id  total_rows_for_cmdb  points_in_window  sum_log_error_count_in_window  max_log_error_count_in_window earliest_timestamp_in_window_where_log.error_count>0               unique_log_names
17         frontend-2                   60                60                          605.0                           27.0                          2022-03-20 21:00:00+08:00    log.error_count,log.row_count
15         frontend-0                   60                60                          548.0                           24.0                          2022-03-20 21:00:00+08:00    log.error_count,log.row_count
16         frontend-1                   60                60                          439.0                           19.0                          2022-03-20 21:00:00+08:00    log.error_count,log.row_count
3       cartservice-0                   60                60                            0.0                            0.0                                                NaT    log.error_count,log.row_count
0         adservice-0                   60                60                            0.0                            0.0                                                NaT    log.error_count,log.row_count
1         adservice-1                   60                60                            0.0                            0.0                                                NaT    log.error_count,log.row_count
2         adservice-2                   60                60                            0.0                            0.0                                                NaT    log.error_count,log.row_count
7   checkoutservice-1                   60                60                            0.0                            0.0                                                NaT    log.error_count,log.row_count
8   checkoutservice-2                   60                60                            0.0                            0.0                                                NaT    log.error_count,log.row_count
9   currencyservice-0                   60                60                            0.0                            0.0                                                NaT    log.error_count,log.row_count
10  currencyservice-1                   60                60                            0.0                            0.0                                                NaT    log.error_count,log.row_count
11  currencyservice-2                   60                60                            0.0                            0.0                                                NaT    log.error_count,log.row_count
4       cartservice-1                   60                60                            0.0                            0.0                                                NaT    log.error_count,log.row_count
5       cartservice-2                   60                60                            0.0                            0.0                                                NaT    log.error_count,log.row_count
6   checkoutservice-0                   60                60                            0.0                            0.0                                                NaT    log.error_count,log.row_count
14     emailservice-2                   60                60                            0.0                            0.0                                                NaT    log.error_count,log.row_count
13     emailservice-1                   60                60                            0.0                            0.0                                                NaT    log.error_count,log.row_count
12     emailservice-0                   60                60                            0.0                            0.0                                                NaT    log.error_count,log.row_count
18   paymentservice-0                   60                60                            0.0                            0.0                                                NaT    log.error_count,log.row_count
19   paymentservice-1                   60                60                            0.0                            0.0                                                NaT    log.error_count,log.row_count)```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

# reuse existing df_error if available, else load
if 'df_error' not in globals():
    df_error = pd.read_csv('error_logs.csv')

# ensure UTC+8 tz column
tz = pytz.timezone('Asia/Shanghai')
if 'dt_local' not in df_error.columns:
    df_error['dt_local'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Candidate frontends and incident window (UTC+8)
frontends = ['frontend-0','frontend-1','frontend-2']
start_window = pd.Timestamp('2022-03-20 21:00:00', tz=tz)
end_window = pd.Timestamp('2022-03-20 21:30:00', tz=tz)  # exclusive

# Filter rows for frontends within window
mask_front = df_error['cmdb_id'].isin(frontends) & (df_error['dt_local'] >= start_window) & (df_error['dt_local'] < end_window)
df_front_win = df_error.loc[mask_front].copy()

# Prepare compact sample table: earliest 10 rows per cmdb_id, sorted by cmdb_id then timestamp ascending
if df_front_win.empty:
    front_samples = pd.DataFrame(columns=['cmdb_id','timestamp','message'])
else:
    df_front_win_sorted = df_front_win.sort_values(['cmdb_id','dt_local'])
    # take up to earliest 10 per cmdb_id
    front_samples = df_front_win_sorted.groupby('cmdb_id', group_keys=False).head(10)[['cmdb_id','dt_local','message']].rename(columns={'dt_local':'timestamp'}).reset_index(drop=True)

# Aggregate distinct message texts with counts for these frontends (top 20 by count)
if df_front_win.empty:
    front_message_counts = pd.DataFrame(columns=['message','count'])
else:
    front_message_counts = (
        df_front_win['message']
        .value_counts(dropna=False)
        .reset_index()
        .rename(columns={'index':'message','message':'count'})
        .head(20)
    )

# Display compact outputs
front_samples, front_message_counts

```
Out[6]:
```
summary = (
    "Summary (frontends, incident window 2022-03-20 21:00:00–21:30:00 UTC+8):\n\n"
    "- Filtered cmdb_ids: frontend-0, frontend-1, frontend-2.\n"
    "- Sample rows: earliest 10 rows per frontend were returned (30 rows total). Timestamps start at ~2022-03-20 21:00:00+08 and progress within the 21:00–21:30 window.\n"
    "- Distinct messages: a single repeated message was observed across these frontends: \"severity: warning, message: failed to retrieve...\". This message appears 1,592 times in the window (it is the dominant/only message for these frontends in this window).\n"
    "- Interpretation: The three frontend pods are the primary sources of error logs during the incident window, repeatedly emitting the same warning. The earliest occurrences are at the start of the window (21:00:00+08).\n"
    "- Recommendation: Investigate the frontends for the cause of the repeated \"failed to retrieve\" warning (check downstream calls, network, and correlating metric/trace spikes around 21:00)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                 timestamp                                            message
0   frontend-0 2022-03-20 21:00:00+08:00  severity: warning, message: failed to retrieve...
1   frontend-0 2022-03-20 21:00:01+08:00  severity: warning, message: failed to retrieve...
2   frontend-0 2022-03-20 21:00:06+08:00  severity: warning, message: failed to retrieve...
3   frontend-0 2022-03-20 21:00:09+08:00  severity: warning, message: failed to retrieve...
4   frontend-0 2022-03-20 21:00:11+08:00  severity: warning, message: failed to retrieve...
..         ...                       ...                                                ...
25  frontend-2 2022-03-20 21:00:09+08:00  severity: warning, message: failed to retrieve...
26  frontend-2 2022-03-20 21:00:12+08:00  severity: warning, message: failed to retrieve...
27  frontend-2 2022-03-20 21:00:13+08:00  severity: warning, message: failed to retrieve...
28  frontend-2 2022-03-20 21:00:15+08:00  severity: warning, message: failed to retrieve...
29  frontend-2 2022-03-20 21:00:17+08:00  severity: warning, message: failed to retrieve...

[30 rows x 3 columns],                                                count  count
0  severity: warning, message: failed to retrieve...   1592)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

# Reuse df_trace, to_local_dt, tz if available from prior steps
import pandas as pd
import pytz
import re

tz = pytz.timezone('Asia/Shanghai')

# Ensure dt_local exists on df_trace
if 'df_trace' not in globals():
    df_trace = pd.read_csv('trace.csv')
if 'dt_local' not in df_trace.columns:
    df_trace['dt_local'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Candidate frontends and filter for trace.to_ or trace.from_ in trace_name
candidates = ['frontend-0','frontend-1','frontend-2','frontend2-0']
mask_candidates = df_trace['cmdb_id'].isin(candidates)
mask_trace_names = df_trace['trace_name'].str.contains(r'trace\.to_|trace\.from_', regex=True)
df_sub = df_trace[mask_candidates & mask_trace_names].copy()

# If no data, prepare empty outputs
start_window = pd.Timestamp('2022-03-20 21:00:00', tz=tz)
end_window = pd.Timestamp('2022-03-20 21:30:00', tz=tz)

if df_sub.shape[0] == 0:
    result_top20 = pd.DataFrame(columns=[
        'cmdb_id','trace_name','total_points','global_p95','global_p5',
        'points_in_window','anomaly_count_above_p95','anomaly_count_below_p5',
        'earliest_anomaly_ts_in_window','avg_value_in_window','max_value_in_window','min_value_in_window'
    ])
    target_summary = pd.DataFrame(columns=['target','earliest_anomaly_ts'])
else:
    # 1) Compute global stats per (cmdb_id, trace_name) using full series BEFORE window filtering
    pairs_global = (
        df_sub
        .groupby(['cmdb_id','trace_name'])['value']
        .agg(
            total_points='count',
            global_p95=lambda s: s.quantile(0.95),
            global_p5 = lambda s: s.quantile(0.05)
        )
        .reset_index()
    )

    # 2) Filter to incident window (UTC+8)
    mask_window = (df_sub['dt_local'] >= start_window) & (df_sub['dt_local'] < end_window)
    df_w = df_sub[mask_window].copy()

    # 3) Merge global thresholds into window rows (so thresholds computed on full series)
    df_w = df_w.merge(pairs_global, on=['cmdb_id','trace_name'], how='left', validate='m:1')

    # 4) Mark anomalies in window using global thresholds
    df_w['anom_above'] = df_w['value'] >= df_w['global_p95']
    df_w['anom_below'] = df_w['value'] <= df_w['global_p5']
    df_w['is_anom'] = df_w['anom_above'] | df_w['anom_below']

    # 5) Aggregate window stats per pair
    grp = df_w.groupby(['cmdb_id','trace_name'])
    window_stats = grp.agg(
        points_in_window = ('value','size'),
        anomaly_count_above_p95 = ('anom_above','sum'),
        anomaly_count_below_p5 = ('anom_below','sum'),
        avg_value_in_window = ('value','mean'),
        max_value_in_window = ('value','max'),
        min_value_in_window = ('value','min')
    ).reset_index()

    # earliest anomaly timestamp where either condition holds
    df_w_anom = df_w[df_w['is_anom']].groupby(['cmdb_id','trace_name'])['dt_local'].min().reset_index().rename(columns={'dt_local':'earliest_anomaly_ts_in_window'})

    window_stats = window_stats.merge(df_w_anom, on=['cmdb_id','trace_name'], how='left', validate='1:1')

    # 6) Combine with global thresholds (only pairs that have points in window)
    combined = pairs_global.merge(window_stats, on=['cmdb_id','trace_name'], how='inner', validate='1:1')

    # 7) Exclude constant series where max == min in the window
    combined_nonconst = combined[combined['max_value_in_window'] != combined['min_value_in_window']].copy()

    # 8) Compute anomaly sum for sorting and sort by anomaly_sum desc, then earliest_anomaly_ts_in_window asc
    combined_nonconst['anomaly_sum'] = combined_nonconst['anomaly_count_above_p95'].fillna(0).astype(int) + combined_nonconst['anomaly_count_below_p5'].fillna(0).astype(int)
    combined_nonconst = combined_nonconst.sort_values(by=['anomaly_sum','earliest_anomaly_ts_in_window'], ascending=[False, True])

    # 9) Select requested columns and top 20
    final_cols = [
        'cmdb_id','trace_name','total_points','global_p95','global_p5',
        'points_in_window','anomaly_count_above_p95','anomaly_count_below_p5',
        'earliest_anomaly_ts_in_window','avg_value_in_window','max_value_in_window','min_value_in_window'
    ]
    result_top20 = combined_nonconst[final_cols].head(20).reset_index(drop=True)

    # 10) Parse downstream target names from trace_name for returned rows and compute earliest anomaly timestamp per target
    # We'll extract 'to_<target>' or 'from_<target>' token
    def extract_target_key(tname):
        m = re.search(r'trace\.(to|from)_(?P<target>[^.]+)', tname)
        if m:
            return f"{m.group(1)}_{m.group('target')}"
        return None

    result_top20['target_key'] = result_top20['trace_name'].apply(extract_target_key)

    # For target summary, consider only rows with a non-null earliest anomaly timestamp
    target_agg = (
        result_top20[['target_key','earliest_anomaly_ts_in_window']]
        .dropna(subset=['target_key'])
        .groupby('target_key', as_index=False)
        .agg(earliest_anomaly_ts = ('earliest_anomaly_ts_in_window','min'))
        .sort_values('earliest_anomaly_ts')
    ).reset_index(drop=True)

    # compact target summary
    target_summary = target_agg.rename(columns={'target_key':'target'})

# Display compact outputs
result_top20, target_summary

```
Out[7]:
```
Summary of trace analysis for frontends (frontend-0, frontend-1, frontend-2, frontend2-0), restricted to trace.from_/trace.to_ series, incident window 2022-03-20 21:00–21:30 (UTC+8).

What I computed
- For each (cmdb_id, trace_name) I computed global thresholds from the full series (global_p95, global_p5), then measured window stats (points_in_window, anomaly counts, earliest anomaly ts, avg/max/min in window).
- I excluded constant series (max == min) and returned the top 20 pairs ranked by total anomaly count (anomaly_count_above_p95 + anomaly_count_below_p5), tie-broken by earliest anomaly timestamp.

Key findings (top rows, compact)
- Many frontend→cartservice duration_p95 series and frontend→checkoutservice row_count series show the largest anomaly counts during the window. Examples (values rounded / paraphrased):
  - frontend-0 → trace.to_cartservice-0.duration_p95: 30 points in window, many anomalies, earliest anomaly 21:00:00, avg ≈ 0.000033s, max ≈ 0.001s, min = 0.
  - frontend-0 → trace.to_cartservice-2.duration_p95: 30 pts, earliest 21:00:00, avg ≈ 0.00005s, max ≈ 0.00085s.
  - frontend-1 → trace.to_adservice-0.duration_mean: 30 pts, anomaly counts high (10 above / 20 below), earliest 21:00:00, avg ≈ 0.000012s.
  - frontend-2 → trace.to_checkoutservice-0.row_count: 29 pts, avg ≈ 1.41, max 3, min 1, anomalies above/below 11/18, earliest 21:00:00.
  - frontend-2 → trace.to_checkoutservice-2.row_count: 29 pts, avg ≈ 1.41, max 2, min 1, anomalies above/below 12/17, earliest 21:00:00.
  - frontend-2 → trace.to_checkoutservice-1.row_count: 30 pts, avg ≈ 1.33, max 2, min 1, anomalies above/below 10/20, earliest 21:00:00.
  - Additional entries include many frontend→cartservice.* and frontend→checkoutservice.* duration_p95 and duration_mean traces with anomalies.

Interpretation
- The top affected downstreams are cartservice and checkoutservice (multiple pod instances), plus some adservice calls. These show the most anomalous behavior from frontends during the incident window.
- Earliest anomalies for the reported targets are essentially at the window start (21:00:00+08) for nearly all targets — indicating issues began at the start of the incident window.
- For row_count traces to checkoutservice pods, the anomalies indicate changes in request counts (avg ~1–1.4, some observations up to 2–3), suggesting traffic/request-pattern changes at frontends to these checkout/cart endpoints.
- Many duration metrics are small absolute values but exhibit deviations from their global thresholds; these may point to latency changes on frontend→backend calls.

Downstream targets observed among the returned rows and earliest anomaly time per target (compact)
- to_adservice-0 — earliest anomaly: 2022-03-20 21:00:00+08
- to_cartservice-0 — earliest anomaly: 2022-03-20 21:00:00+08
- to_cartservice-1 — earliest anomaly: 2022-03-20 21:00:00+08
- to_cartservice-2 — earliest anomaly: 2022-03-20 21:00:00+08
- to_checkoutservice-0 — earliest anomaly: 2022-03-20 21:00:00+08
- to_checkoutservice-1 — earliest anomaly: 2022-03-20 21:00:00+08
- to_checkoutservice-2 — earliest anomaly: 2022-03-20 21:00:00+08

Recommended next steps
1. Prioritize investigating frontend → cartservice and frontend → checkoutservice interactions around 21:00. Check backend (cart/checkout) service health, latency, and request handling at that time.  
2. Correlate these trace anomalies with the frontend error_logs (repeated "failed to retrieve..." warnings) and the metric anomalies previously found for frontend2-0 and adservice/adservice2.  
3. If desired, I can (a) show the full top-20 trace rows verbatim, (b) remove series where global thresholds collapse to zero and re-rank, or (c) cross-correlate these trace timestamps with specific metric spikes. Which would you like next?

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                            trace_name  total_points  global_p95  global_p5  points_in_window  anomaly_count_above_p95  anomaly_count_below_p5 earliest_anomaly_ts_in_window  avg_value_in_window  max_value_in_window  min_value_in_window            target_key
0   frontend-0   trace.to_cartservice-0.duration_p95            30    0.000000   0.000000                30                       30                      29     2022-03-20 21:00:00+08:00             0.000033             0.001000             0.000000      to_cartservice-0
1   frontend-0   trace.to_cartservice-2.duration_p95            30    0.000357   0.000000                30                        2                      28     2022-03-20 21:00:00+08:00             0.000050             0.000850             0.000000      to_cartservice-2
2   frontend-1    trace.to_adservice-0.duration_mean            30    0.000013   0.000012                30                       10                      20     2022-03-20 21:00:00+08:00             0.000012             0.000018             0.000011        to_adservice-0
3   frontend-1   trace.to_cartservice-0.duration_p95            30    0.000027   0.000000                30                        2                      28     2022-03-20 21:00:00+08:00             0.000033             0.000950             0.000000      to_cartservice-0
4   frontend-2   trace.to_cartservice-0.duration_p95            30    0.000302   0.000000                30                        2                      28     2022-03-20 21:00:00+08:00             0.000042             0.000700             0.000000      to_cartservice-0
5   frontend-2  trace.to_checkoutservice-1.row_count            30    2.000000   1.000000                30                       10                      20     2022-03-20 21:00:00+08:00             1.333333             2.000000             1.000000  to_checkoutservice-1
6   frontend-0   trace.to_cartservice-1.duration_p95            30    0.000627   0.000000                30                        2                      27     2022-03-20 21:00:00+08:00             0.000067             0.000750             0.000000      to_cartservice-1
7   frontend-1   trace.to_cartservice-1.duration_p95            30    0.000132   0.000000                30                        2                      27     2022-03-20 21:00:00+08:00             0.000042             0.001000             0.000000      to_cartservice-1
8   frontend-2   trace.to_cartservice-1.duration_p95            30    0.000477   0.000000                30                        2                      27     2022-03-20 21:00:00+08:00             0.000048             0.000500             0.000000      to_cartservice-1
9   frontend-2  trace.to_checkoutservice-0.row_count            29    2.000000   1.000000                29                       11                      18     2022-03-20 21:00:00+08:00             1.413793             3.000000             1.000000  to_checkoutservice-0
10  frontend-2  trace.to_checkoutservice-2.row_count            29    2.000000   1.000000                29                       12                      17     2022-03-20 21:00:00+08:00             1.413793             2.000000             1.000000  to_checkoutservice-2
11  frontend-1   trace.to_cartservice-2.duration_p95            30    0.000155   0.000000                30                        2                      26     2022-03-20 21:00:00+08:00             0.000043             0.000950             0.000000      to_cartservice-2
12  frontend-0  trace.to_checkoutservice-1.row_count            28    2.000000   1.000000                28                        8                      20     2022-03-20 21:01:00+08:00             1.285714             2.000000             1.000000  to_checkoutservice-1
13  frontend-0  trace.to_checkoutservice-0.row_count            27    2.000000   1.000000                27                       10                      17     2022-03-20 21:00:00+08:00             1.370370             2.000000             1.000000  to_checkoutservice-0
14  frontend-1  trace.to_checkoutservice-0.row_count            27    2.000000   1.000000                27                        3                      24     2022-03-20 21:00:00+08:00             1.111111             2.000000             1.000000  to_checkoutservice-0
15  frontend-1  trace.to_checkoutservice-1.row_count            27    1.700000   1.000000                27                        2                      25     2022-03-20 21:00:00+08:00             1.074074             2.000000             1.000000  to_checkoutservice-1
16  frontend-1  trace.to_checkoutservice-2.row_count            26    2.000000   1.000000                26                        4                      22     2022-03-20 21:00:00+08:00             1.153846             2.000000             1.000000  to_checkoutservice-2
17  frontend-0  trace.to_checkoutservice-2.row_count            25    2.000000   1.000000                25                       11                      14     2022-03-20 21:00:00+08:00             1.440000             2.000000             1.000000  to_checkoutservice-2
18  frontend-2    trace.to_adservice-0.duration_mean            30    0.000018   0.000012                30                        2                      22     2022-03-20 21:00:00+08:00             0.000013             0.000023             0.000012        to_adservice-0
19  frontend-1  trace.to_cartservice-0.duration_mean            30    0.000049   0.000000                30                        2                      22     2022-03-20 21:01:00+08:00             0.000013             0.000091             0.000000      to_cartservice-0,                  target       earliest_anomaly_ts
0        to_adservice-0 2022-03-20 21:00:00+08:00
1      to_cartservice-0 2022-03-20 21:00:00+08:00
2      to_cartservice-1 2022-03-20 21:00:00+08:00
3      to_cartservice-2 2022-03-20 21:00:00+08:00
4  to_checkoutservice-0 2022-03-20 21:00:00+08:00
5  to_checkoutservice-1 2022-03-20 21:00:00+08:00
6  to_checkoutservice-2 2022-03-20 21:00:00+08:00)```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing df_metric if available; else load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Ensure dt_local exists (localized to UTC+8)
if 'dt_local' not in df_metric.columns:
    df_metric['dt_local'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Candidate cmdb_ids to restrict
candidates = [
    'frontend-0','frontend-1','frontend-2','frontend2-0',
    'cartservice-0','cartservice-1','cartservice-2','cartservice2-0',
    'checkoutservice-0','checkoutservice-1','checkoutservice-2','checkoutservice2-0'
]

# Restrict metric series to these cmdb_ids (global thresholds computed on full available series for these ids)
df_cand = df_metric[df_metric['cmdb_id'].isin(candidates)].copy()

# If no data, prepare empty outputs
start_window = pd.Timestamp('2022-03-20 21:00:00', tz=tz)
end_window = pd.Timestamp('2022-03-20 21:30:00', tz=tz)  # exclusive

if df_cand.shape[0] == 0:
    result_top20 = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','total_points','global_p95','global_p5',
        'points_in_window','anomaly_count_above_p95','anomaly_count_below_p5',
        'earliest_anomaly_ts_in_window','max_value_in_window','min_value_in_window',
        'std_in_window','deviation_score'
    ])
    per_cmdb_agg = pd.DataFrame(columns=['cmdb_id','sum_of_anomaly_counts','earliest_anomaly_ts_among_KPIs'])
else:
    # 1) Compute global stats per (cmdb_id, kpi_name) using full series BEFORE filtering
    pairs = (
        df_cand
        .groupby(['cmdb_id','kpi_name'])['value']
        .agg(
            total_points = 'count',
            global_p95 = lambda s: s.quantile(0.95),
            global_p5  = lambda s: s.quantile(0.05)
        )
        .reset_index()
    )

    # 2) Filter rows to the incident window
    mask_window = (df_cand['dt_local'] >= start_window) & (df_cand['dt_local'] < end_window)
    df_w = df_cand[mask_window].copy()

    # If no rows in window, produce empty outputs accordingly
    if df_w.shape[0] == 0:
        result_top20 = pd.DataFrame(columns=[
            'cmdb_id','kpi_name','total_points','global_p95','global_p5',
            'points_in_window','anomaly_count_above_p95','anomaly_count_below_p5',
            'earliest_anomaly_ts_in_window','max_value_in_window','min_value_in_window',
            'std_in_window','deviation_score'
        ])
        per_cmdb_agg = pd.DataFrame({
            'cmdb_id': candidates,
            'sum_of_anomaly_counts': [0]*len(candidates),
            'earliest_anomaly_ts_among_KPIs': [pd.NaT]*len(candidates)
        })
    else:
        # 3) Merge global thresholds into window rows
        df_w = df_w.merge(pairs, on=['cmdb_id','kpi_name'], how='left', validate='m:1')

        # 4) Mark anomalies in window using global thresholds
        df_w['anom_above'] = df_w['value'] >= df_w['global_p95']
        df_w['anom_below'] = df_w['value'] <= df_w['global_p5']
        df_w['is_anom'] = df_w['anom_above'] | df_w['anom_below']

        # 5) Aggregate per-pair in window
        grp = df_w.groupby(['cmdb_id','kpi_name'])
        window_stats = grp.agg(
            points_in_window = ('value','size'),
            anomaly_count_above_p95 = ('anom_above','sum'),
            anomaly_count_below_p5 = ('anom_below','sum'),
            max_value_in_window = ('value','max'),
            min_value_in_window = ('value','min'),
            std_in_window = ('value','std')
        ).reset_index()

        # earliest anomaly timestamp per pair (where is_anom True)
        df_w_anom = df_w[df_w['is_anom']].groupby(['cmdb_id','kpi_name'])['dt_local'].min().reset_index().rename(columns={'dt_local':'earliest_anomaly_ts_in_window'})

        window_stats = window_stats.merge(df_w_anom, on=['cmdb_id','kpi_name'], how='left', validate='1:1')

        # 6) Combine with global thresholds (only pairs with points in window)
        combined = pairs.merge(window_stats, on=['cmdb_id','kpi_name'], how='inner', validate='1:1')

        # 7) Flag constant series (max==min) and compute deviation_score
        combined['constant_series'] = combined['max_value_in_window'] == combined['min_value_in_window']
        # compute positive deltas, else 0
        delta_above = (combined['max_value_in_window'] - combined['global_p95']).clip(lower=0)
        delta_below = (combined['global_p5'] - combined['min_value_in_window']).clip(lower=0)
        combined['deviation_score'] = np.maximum(delta_above.fillna(0), delta_below.fillna(0))

        # 8) Exclude constant series True
        combined_nonconst = combined[~combined['constant_series']].copy()

        # 9) Compute anomaly sum and sort by anomaly_sum desc, then deviation_score desc
        combined_nonconst['anomaly_sum'] = combined_nonconst['anomaly_count_above_p95'].fillna(0).astype(int) + combined_nonconst['anomaly_count_below_p5'].fillna(0).astype(int)
        combined_nonconst_sorted = combined_nonconst.sort_values(by=['anomaly_sum','deviation_score'], ascending=[False, False])

        # 10) Prepare final table (top 20) with requested columns
        final_cols = [
            'cmdb_id','kpi_name','total_points','global_p95','global_p5',
            'points_in_window','anomaly_count_above_p95','anomaly_count_below_p5',
            'earliest_anomaly_ts_in_window','max_value_in_window','min_value_in_window',
            'std_in_window','deviation_score'
        ]
        result_top20 = combined_nonconst_sorted[final_cols].head(20).reset_index(drop=True)

        # 11) Per-cmdb_id aggregate across all non-constant KPIs (for the listed cmdb_ids)
        per_kpi_anom = combined_nonconst[['cmdb_id','anomaly_sum','earliest_anomaly_ts_in_window']].copy()
        per_cmdb = per_kpi_anom.groupby('cmdb_id', as_index=False).agg(
            sum_of_anomaly_counts = ('anomaly_sum','sum'),
            earliest_anomaly_ts_among_KPIs = ('earliest_anomaly_ts_in_window','min')
        )

        # Ensure all listed cmdb_ids are present in aggregate (fill missing with zeros/NaT)
        per_cmdb_agg = pd.DataFrame({'cmdb_id': candidates}).merge(per_cmdb, on='cmdb_id', how='left')
        per_cmdb_agg['sum_of_anomaly_counts'] = per_cmdb_agg['sum_of_anomaly_counts'].fillna(0).astype(int)
        per_cmdb_agg['earliest_anomaly_ts_among_KPIs'] = per_cmdb_agg['earliest_anomaly_ts_among_KPIs'].dt.tz_convert(tz)

# Final compact outputs (top 20 KPI rows and per-cmdb aggregate for listed ids)
result_top20, per_cmdb_agg

```
Out[8]:
```
Summary (metrics for selected frontends, cartservice, checkoutservice pods; incident window 2022-03-20 21:00–21:30 UTC+8)

What I computed
- For the listed cmdb_ids I computed global p95/p5 from the full series, then measured window statistics and flagged anomalies where window values exceeded those global thresholds.
- Constant-valued KPIs (max==min in window) were excluded from the returned list.
- Returned the top 20 (cmdb_id, kpi_name) pairs ranked by total anomaly counts (above_p95 + below_p5), tie-broken by deviation.

Key findings (top anomalous KPIs, compact)
- Many of the top anomalies are container file-descriptor and socket metrics on node-5 / node-6 and appear across checkoutservice, cartservice and frontend pods:
  - checkoutservice-0 / container_file_descriptors: 30 pts, anomaly_count_above = 29, below = 30, earliest = 2022-03-20 21:00:00, max = 9, min = 8.5, std ≈ 0.091.
  - checkoutservice-0 / container_sockets: similar behavior (max 3, min 2.5).
  - frontend-2 / container_file_descriptors and /container_sockets: heavy anomaly counts, earliest 21:00, non-constant.
  - cartservice2-0 and cartservice-2 show similar file_descriptor/socket anomalies on node-6 / node-5.
- Several mesh/jaeger collector trace-related metrics for checkoutservice and checkoutservice2-0 appear among top rows (non-zero p95, some anomalies).
- frontend2-0 shows notable container memory and CPU throttling metrics:
  - container_memory_mapped_file (non-constant, significant std and deviation)
  - container_cpu_cfs_throttled_pct / periods (some points above p95)
- Some checkoutservice* jaeger/mesh metrics and container_cpu_cfs_throttled_... also show anomalous counts and positive deviation scores.
- All earliest anomalies for the reported KPIs are at or very near the window start: 2022-03-20 21:00:00+08 (a few entries earliest at 21:03 for one checkoutservice-1 jaeger metric).

Per-cmdb aggregate summary (sum of anomaly counts and earliest anomaly time)
- frontend2-0: sum_of_anomaly_counts = 412, earliest anomaly 2022-03-20 21:00:00+08
- checkoutservice-0: 507, earliest 21:00:00+08
- checkoutservice-1: 449, earliest 21:00:00+08
- checkoutservice2-0: 420, earliest 21:00:00+08
- frontend-2: 381, earliest 21:00:00+08
- cartservice2-0: 219, earliest 21:00:00+08
- checkoutservice-2: 325, earliest 21:00:00+08
- cartservice-2: 197, earliest 21:00:00+08
- frontend-0: 366, earliest 21:00:00+08
- frontend-1: 358, earliest 21:00:00+08
- cartservice-0: 141, earliest 21:00:00+08
- cartservice-1: 143, earliest 21:00:00+08

Interpretation and recommended next steps
- The anomaly signals are concentrated around container-level metrics (file descriptors, sockets, CPU throttling, memory mapping) on the nodes/pods that host frontends, cartservice and checkoutservice pods — suggesting resource or networking pressure at the container/node level starting at 21:00.
- Checkoutservice and frontend2-0 are among the most affected (highest summed anomaly counts).
- Because earliest anomalies align with the window start (21:00), investigate events just prior to or at 21:00:00: check node resource usage, container limits/throttling, and any deployment/scale/GC activity.
- Correlate these metric anomalies with the earlier trace-level findings (frontend→cart/checkout latency/row_count changes) and the frontend error logs ("failed to retrieve...") to pinpoint whether backend saturation, resource exhaustion, or network/connectivity issues caused frontend failures.

If you want, I can:
- show the detailed top-20 table rows verbatim, or
- expand to include the constant-series KPIs, or
- correlate the top anomalous KPIs timestamps with node/container-level events or logs. Which would you like next?

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id                                           kpi_name  total_points    global_p95  global_p5  points_in_window  anomaly_count_above_p95  anomaly_count_below_p5 earliest_anomaly_ts_in_window  max_value_in_window  min_value_in_window  std_in_window  deviation_score
0    checkoutservice-0        container.node-5.container_file_descriptors            30  9.000000e+00        9.0                30                       29                      30     2022-03-20 21:00:00+08:00         9.000000e+00                  8.5       0.091287     5.000000e-01
1    checkoutservice-0                 container.node-5.container_sockets            30  3.000000e+00        3.0                30                       29                      30     2022-03-20 21:00:00+08:00         3.000000e+00                  2.5       0.091287     5.000000e-01
2   checkoutservice2-0  mesh.source.checkoutservice2.jaeger-collector....            30  5.250000e-04        0.0                30                        5                      25     2022-03-20 21:00:00+08:00         1.050000e-03                  0.0       0.000254     5.250000e-04
3           frontend-2        container.node-5.container_file_descriptors            30  2.400000e+01       19.0                30                       15                      15     2022-03-20 21:00:00+08:00         2.400000e+01                 19.0       2.542738     0.000000e+00
4           frontend-2                 container.node-5.container_sockets            30  1.800000e+01       13.0                30                       15                      15     2022-03-20 21:00:00+08:00         1.800000e+01                 13.0       2.542738     0.000000e+00
5    checkoutservice-0  mesh.source.checkoutservice.jaeger-collector.i...            30  5.250000e-04        0.0                30                        3                      25     2022-03-20 21:00:00+08:00         5.250000e-04                  0.0       0.000199     2.999987e-16
6          frontend2-0      container.node-6.container_memory_mapped_file            30  6.782976e+06  5787648.0                30                       18                       9     2022-03-20 21:00:00+08:00         7.118848e+06            5787648.0  493523.725234     3.358720e+05
7          frontend2-0  container.node-6.container_cpu_cfs_throttled_p...            30  1.000000e+00        0.0                30                        3                      24     2022-03-20 21:00:00+08:00         1.666667e+00                  0.0       0.398753     6.666667e-01
8       cartservice2-0        container.node-6.container_file_descriptors            30  1.630000e+02      162.0                30                       21                       6     2022-03-20 21:00:00+08:00         1.630000e+02                162.0       0.405147     0.000000e+00
9       cartservice2-0                 container.node-6.container_sockets            30  8.000000e+00        7.0                30                       21                       6     2022-03-20 21:00:00+08:00         8.000000e+00                  7.0       0.405147     0.000000e+00
10   checkoutservice-1  mesh.source.checkoutservice.jaeger-collector.i...            30  1.326000e-02        0.0                30                        2                      21     2022-03-20 21:03:00+08:00         2.325000e-02                  0.0       0.005861     9.990000e-03
11       cartservice-2        container.node-5.container_file_descriptors            30  1.630000e+02      162.0                30                       11                      12     2022-03-20 21:00:00+08:00         1.630000e+02                162.0       0.444959     0.000000e+00
12       cartservice-2                 container.node-5.container_sockets            30  8.000000e+00        7.0                30                       11                      12     2022-03-20 21:00:00+08:00         8.000000e+00                  7.0       0.444959     0.000000e+00
13          frontend-0        container.node-5.container_file_descriptors            30  2.500000e+01       20.0                30                        8                      15     2022-03-20 21:00:00+08:00         2.500000e+01                 20.0       2.347718     0.000000e+00
14          frontend-0                 container.node-5.container_sockets            30  1.900000e+01       14.0                30                        8                      15     2022-03-20 21:00:00+08:00         1.900000e+01                 14.0       2.347718     0.000000e+00
15  checkoutservice2-0  container.node-5.container_cpu_cfs_throttled_p...            30  1.000000e+00        0.0                30                        8                      12     2022-03-20 21:00:00+08:00         1.500000e+00                  0.0       0.442368     5.000000e-01
16          frontend-1        container.node-5.container_file_descriptors            30  2.300000e+01       19.0                30                       15                       5     2022-03-20 21:00:00+08:00         2.300000e+01                 19.0       1.711691     0.000000e+00
17          frontend-1                 container.node-5.container_sockets            30  1.700000e+01       13.0                30                       15                       5     2022-03-20 21:00:00+08:00         1.700000e+01                 13.0       1.711691     0.000000e+00
18   checkoutservice-2  mesh.source.checkoutservice.jaeger-collector.i...            30  2.375000e-02        0.0                30                        3                      14     2022-03-20 21:00:00+08:00         3.602500e-02                  0.0       0.008593     1.227500e-02
19  checkoutservice2-0  container.node-5.container_cpu_cfs_throttled_s...            30  6.775679e-02        0.0                30                        2                      14     2022-03-20 21:00:00+08:00         6.998992e-02                  0.0       0.020548     2.233124e-03,                cmdb_id  sum_of_anomaly_counts earliest_anomaly_ts_among_KPIs
0           frontend-0                    366      2022-03-20 21:00:00+08:00
1           frontend-1                    358      2022-03-20 21:00:00+08:00
2           frontend-2                    381      2022-03-20 21:00:00+08:00
3          frontend2-0                    412      2022-03-20 21:00:00+08:00
4        cartservice-0                    141      2022-03-20 21:00:00+08:00
5        cartservice-1                    143      2022-03-20 21:00:00+08:00
6        cartservice-2                    197      2022-03-20 21:00:00+08:00
7       cartservice2-0                    219      2022-03-20 21:00:00+08:00
8    checkoutservice-0                    507      2022-03-20 21:00:00+08:00
9    checkoutservice-1                    449      2022-03-20 21:00:00+08:00
10   checkoutservice-2                    325      2022-03-20 21:00:00+08:00
11  checkoutservice2-0                    420      2022-03-20 21:00:00+08:00)```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz

# Reuse df_metric and dt_local if present; else load and prepare
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

tz = pytz.timezone('Asia/Shanghai')
if 'dt_local' not in df_metric.columns:
    df_metric['dt_local'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Candidates and KPI filters
candidates = [
    'checkoutservice-0','checkoutservice-1','checkoutservice-2','checkoutservice2-0',
    'frontend-0','frontend-1','frontend-2','frontend2-0'
]
kpi_filters = ['container_file_descriptors','container_sockets']

# Incident window (UTC+8)
start_window = pd.Timestamp('2022-03-20 21:00:00', tz=tz)
end_window = pd.Timestamp('2022-03-20 21:30:00', tz=tz)  # exclusive

# Filter dataset to candidates, kpi_name containing desired substrings, and window
mask_cand = df_metric['cmdb_id'].isin(candidates)
mask_kpi = df_metric['kpi_name'].str.contains('|'.join(kpi_filters), regex=True)
mask_window = (df_metric['dt_local'] >= start_window) & (df_metric['dt_local'] < end_window)

df_sel = df_metric[mask_cand & mask_kpi & mask_window].copy()

# (a) Earliest 10 rows per (cmdb_id, kpi_name) in window, sorted by timestamp asc
if df_sel.empty:
    samples_df = pd.DataFrame(columns=['cmdb_id','kpi_name','timestamp','value'])
else:
    df_sel_sorted = df_sel.sort_values(['cmdb_id','kpi_name','dt_local'])
    samples_df = (
        df_sel_sorted
        .groupby(['cmdb_id','kpi_name'], group_keys=False)
        .head(10)[['cmdb_id','kpi_name','dt_local','value']]
        .rename(columns={'dt_local':'timestamp'})
        .reset_index(drop=True)
    )

# (b) Aggregate per (cmdb_id, kpi_name)
if df_sel.empty:
    agg_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','points_in_window','min_value_in_window','max_value_in_window',
        'earliest_timestamp_where_value==max','earliest_timestamp_where_value==min'
    ])
else:
    grp = df_sel.groupby(['cmdb_id','kpi_name'], sort=True)
    agg_list = []
    for (cmdb, kpi), g in grp:
        points = int(g.shape[0])
        min_val = float(g['value'].min())
        max_val = float(g['value'].max())
        # earliest timestamp where value == max
        ts_max = g[g['value'] == max_val]['dt_local'].min()
        ts_min = g[g['value'] == min_val]['dt_local'].min()
        agg_list.append({
            'cmdb_id': cmdb,
            'kpi_name': kpi,
            'points_in_window': points,
            'min_value_in_window': min_val,
            'max_value_in_window': max_val,
            'earliest_timestamp_where_value==max': ts_max,
            'earliest_timestamp_where_value==min': ts_min
        })
    agg_df = pd.DataFrame(agg_list).sort_values(['cmdb_id','kpi_name']).reset_index(drop=True)

# Final compact outputs: samples (earliest 10 per series) and aggregate table for all listed pairs
samples_df, agg_df

```
Out[9]:
```
Summary for container_file_descriptors and container_sockets (window 2022‑03‑20 21:00:00–21:30:00 UTC+8; cmdb_ids: checkoutservice-*, frontend-*, frontend2-0)

What I returned
- Earliest up to 10 rows per (cmdb_id, kpi_name) within the window (sample table).
- A compact aggregate per (cmdb_id, kpi_name) with points_in_window, min, max, and earliest timestamps where the max/min first occurred. (All listed pairs shown; most series have 30 points in-window.)

Key observations (most relevant values shown)
- Frontend pods show noticeably higher file-descriptor and socket counts than checkout pods:
  - frontend-0: file_descriptors max=25 → min=20 (earliest max at 21:00, earliest min at 21:15)
  - frontend-1: file_descriptors max=23 → min=19 (earliest max 21:00, earliest min 21:25)
  - frontend-2: file_descriptors max=24 → min=19 (earliest max 21:00, earliest min 21:15)
  - Corresponding socket counts follow the same pattern (frontend sockets max ~14–19, min ~13–17).
- Some checkoutservice instances have low and largely constant values:
  - checkoutservice-1 file_descriptors = 10.0 constant (max==min at 21:00).
  - checkoutservice-2, checkoutservice2-0 and many checkoutservice kpis are constant at low values (e.g., file_descriptors 9–10, sockets 3–4).
- checkoutservice-0 shows slight variation (file_descriptors max=9.0 at 21:00, min=8.5 at 21:21; sockets max=3.0 → min=2.5).
- frontend2-0 (on node-6) appears constant for these KPIs in the window (file_descriptors = 18, sockets = 12).
- Pattern: for several non-constant series the highest values occur at or near the window start (21:00), with lower values later in the window (often around 21:15–21:25).

Interpretation and recommended next steps
- Frontends had higher connection/resource use and exhibit a decline in file-descriptors/sockets after ~21:00–21:15. This could indicate connection churn, drops, or reduced incoming traffic, or cleanup/close of connections mid-window.
- Checkout pods mostly show small, stable FD/socket counts; checkoutservice-0 shows a small decline similar to frontends but at lower absolute counts.
- Actions to investigate:
  1) Check network/connection-related logs on frontend pods for events around 21:00–21:15 (connection resets, closes, or upstream failures).
  2) Correlate the timestamped drops with frontend error logs ("failed to retrieve...") and the trace anomalies already observed for frontend→checkout/cart calls.
  3) Verify whether container limits, ephemeral port exhaustion, or load-balancer behavior caused connection churn.
  4) If needed, inspect host/node metrics for socket/file-descriptor limits and network errors at the same times.

If you want, I can (a) show the earliest 10 rows for a specific (cmdb_id,kpi_name) pair, or (b) expand to other related KPIs (e.g., connection states, TCP retransmits) to further diagnose connection issues. Which do you prefer?

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id                                     kpi_name                 timestamp  value
0    checkoutservice-0  container.node-5.container_file_descriptors 2022-03-20 21:00:00+08:00    9.0
1    checkoutservice-0  container.node-5.container_file_descriptors 2022-03-20 21:01:00+08:00    9.0
2    checkoutservice-0  container.node-5.container_file_descriptors 2022-03-20 21:02:00+08:00    9.0
3    checkoutservice-0  container.node-5.container_file_descriptors 2022-03-20 21:03:00+08:00    9.0
4    checkoutservice-0  container.node-5.container_file_descriptors 2022-03-20 21:04:00+08:00    9.0
..                 ...                                          ...                       ...    ...
155        frontend2-0           container.node-6.container_sockets 2022-03-20 21:05:00+08:00   12.0
156        frontend2-0           container.node-6.container_sockets 2022-03-20 21:06:00+08:00   12.0
157        frontend2-0           container.node-6.container_sockets 2022-03-20 21:07:00+08:00   12.0
158        frontend2-0           container.node-6.container_sockets 2022-03-20 21:08:00+08:00   12.0
159        frontend2-0           container.node-6.container_sockets 2022-03-20 21:09:00+08:00   12.0

[160 rows x 4 columns],                cmdb_id                                     kpi_name  points_in_window  min_value_in_window  max_value_in_window earliest_timestamp_where_value==max earliest_timestamp_where_value==min
0    checkoutservice-0  container.node-5.container_file_descriptors                30                  8.5                  9.0           2022-03-20 21:00:00+08:00           2022-03-20 21:21:00+08:00
1    checkoutservice-0           container.node-5.container_sockets                30                  2.5                  3.0           2022-03-20 21:00:00+08:00           2022-03-20 21:21:00+08:00
2    checkoutservice-1  container.node-5.container_file_descriptors                30                 10.0                 10.0           2022-03-20 21:00:00+08:00           2022-03-20 21:00:00+08:00
3    checkoutservice-1           container.node-5.container_sockets                30                  4.0                  4.0           2022-03-20 21:00:00+08:00           2022-03-20 21:00:00+08:00
4    checkoutservice-2  container.node-5.container_file_descriptors                30                  9.0                  9.0           2022-03-20 21:00:00+08:00           2022-03-20 21:00:00+08:00
5    checkoutservice-2           container.node-5.container_sockets                30                  3.0                  3.0           2022-03-20 21:00:00+08:00           2022-03-20 21:00:00+08:00
6   checkoutservice2-0  container.node-5.container_file_descriptors                30                  9.0                  9.0           2022-03-20 21:00:00+08:00           2022-03-20 21:00:00+08:00
7   checkoutservice2-0           container.node-5.container_sockets                30                  3.0                  3.0           2022-03-20 21:00:00+08:00           2022-03-20 21:00:00+08:00
8           frontend-0  container.node-5.container_file_descriptors                30                 20.0                 25.0           2022-03-20 21:00:00+08:00           2022-03-20 21:15:00+08:00
9           frontend-0           container.node-5.container_sockets                30                 14.0                 19.0           2022-03-20 21:00:00+08:00           2022-03-20 21:15:00+08:00
10          frontend-1  container.node-5.container_file_descriptors                30                 19.0                 23.0           2022-03-20 21:00:00+08:00           2022-03-20 21:25:00+08:00
11          frontend-1           container.node-5.container_sockets                30                 13.0                 17.0           2022-03-20 21:00:00+08:00           2022-03-20 21:25:00+08:00
12          frontend-2  container.node-5.container_file_descriptors                30                 19.0                 24.0           2022-03-20 21:00:00+08:00           2022-03-20 21:15:00+08:00
13          frontend-2           container.node-5.container_sockets                30                 13.0                 18.0           2022-03-20 21:00:00+08:00           2022-03-20 21:15:00+08:00
14         frontend2-0  container.node-6.container_file_descriptors                30                 18.0                 18.0           2022-03-20 21:00:00+08:00           2022-03-20 21:00:00+08:00
15         frontend2-0           container.node-6.container_sockets                30                 12.0                 12.0           2022-03-20 21:00:00+08:00           2022-03-20 21:00:00+08:00)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import pytz
import re

# Reuse existing df_metric and dt_local when available
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

tz = pytz.timezone('Asia/Shanghai')
if 'dt_local' not in df_metric.columns:
    df_metric['dt_local'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure dt_local is tz-aware in UTC+8
    df_metric['dt_local'] = pd.to_datetime(df_metric['dt_local']).dt.tz_convert(tz)

# Candidate cmdb_ids and node pattern inclusion
candidates = [
    'frontend-0','frontend-1','frontend-2','frontend2-0',
    'checkoutservice-0','checkoutservice-1','checkoutservice-2','checkoutservice2-0'
]

# KPI keyword filter (case-insensitive)
keywords = r'network|transmit|receive|retransmit|err|packet|drop|tcp|rx|tx'
kw_mask = df_metric['kpi_name'].astype(str).str.contains(keywords, case=False, regex=True)

# node-level pattern: kpi_name contains 'node-' followed by digit(s) (e.g., container.node-5...)
node_mask = df_metric['kpi_name'].astype(str).str.contains(r'node-\d+', regex=True)

# include rows where cmdb_id in candidates OR kpi_name contains node-<id>, AND match keyword filter
mask_rows = (df_metric['cmdb_id'].isin(candidates) | node_mask) & kw_mask
df_net = df_metric[mask_rows].copy()

# If no matching data, prepare empty outputs
start_window = pd.Timestamp('2022-03-20 21:00:00', tz=tz)
end_window = pd.Timestamp('2022-03-20 21:30:00', tz=tz)  # exclusive

if df_net.shape[0] == 0:
    result_top20 = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','total_points','global_p95','global_p5',
        'points_in_window','anomaly_count_above_p95','anomaly_count_below_p5',
        'earliest_anomaly_ts_in_window','avg_value_in_window','max_value_in_window','min_value_in_window'
    ])
    per_cmdb_anom = pd.DataFrame(columns=['cmdb_id','earliest_anomaly_ts'])
else:
    # 1) Compute global thresholds on full series (BEFORE filtering to window)
    pairs_global = (
        df_net
        .groupby(['cmdb_id','kpi_name'])['value']
        .agg(
            total_points='count',
            global_p95=lambda s: s.quantile(0.95),
            global_p5 = lambda s: s.quantile(0.05)
        )
        .reset_index()
    )

    # 2) Filter df_net to incident window
    mask_window = (df_net['dt_local'] >= start_window) & (df_net['dt_local'] < end_window)
    df_w = df_net[mask_window].copy()

    # 3) Merge thresholds into window rows
    df_w = df_w.merge(pairs_global, on=['cmdb_id','kpi_name'], how='left', validate='m:1')

    # 4) Mark anomalies using global thresholds
    df_w['anom_above'] = df_w['value'] >= df_w['global_p95']
    df_w['anom_below'] = df_w['value'] <= df_w['global_p5']
    df_w['is_anom'] = df_w['anom_above'] | df_w['anom_below']

    # 5) Aggregate per (cmdb_id,kpi_name) within window
    grp = df_w.groupby(['cmdb_id','kpi_name'])
    window_stats = grp.agg(
        points_in_window = ('value','size'),
        anomaly_count_above_p95 = ('anom_above','sum'),
        anomaly_count_below_p5 = ('anom_below','sum'),
        avg_value_in_window = ('value','mean'),
        max_value_in_window = ('value','max'),
        min_value_in_window = ('value','min')
    ).reset_index()

    # earliest anomaly timestamp per pair (where is_anom True)
    df_w_anom = df_w[df_w['is_anom']].groupby(['cmdb_id','kpi_name'])['dt_local'].min().reset_index().rename(columns={'dt_local':'earliest_anomaly_ts_in_window'})

    window_stats = window_stats.merge(df_w_anom, on=['cmdb_id','kpi_name'], how='left', validate='1:1')

    # 6) Combine global thresholds with window aggregates (only pairs that have points in window)
    combined = pairs_global.merge(window_stats, on=['cmdb_id','kpi_name'], how='inner', validate='1:1')

    # 7) Exclude constant series (max == min)
    combined_nonconst = combined[combined['max_value_in_window'] != combined['min_value_in_window']].copy()

    # 8) Compute anomaly sum and sort by anomaly sum desc then earliest anomaly asc
    combined_nonconst['anomaly_sum'] = combined_nonconst['anomaly_count_above_p95'].fillna(0).astype(int) + combined_nonconst['anomaly_count_below_p5'].fillna(0).astype(int)
    combined_nonconst = combined_nonconst.sort_values(by=['anomaly_sum','earliest_anomaly_ts_in_window'], ascending=[False, True])

    # 9) Select top 20 and desired columns
    final_cols = [
        'cmdb_id','kpi_name','total_points','global_p95','global_p5',
        'points_in_window','anomaly_count_above_p95','anomaly_count_below_p5',
        'earliest_anomaly_ts_in_window','avg_value_in_window','max_value_in_window','min_value_in_window'
    ]
    result_top20 = combined_nonconst[final_cols].head(20).reset_index(drop=True)

    # 10) Per-cmdb_id list of distinct cmdb_ids with network-related anomalies and earliest anomaly ts per cmdb_id
    # consider anomaly_sum>0
    per_cmdb = combined_nonconst[combined_nonconst['anomaly_sum']>0].groupby('cmdb_id', as_index=False).agg(
        earliest_anomaly_ts = ('earliest_anomaly_ts_in_window','min')
    )
    # ensure tz and sort
    per_cmdb['earliest_anomaly_ts'] = pd.to_datetime(per_cmdb['earliest_anomaly_ts']).dt.tz_convert(tz)
    per_cmdb_anom = per_cmdb.sort_values('earliest_anomaly_ts').reset_index(drop=True)

# Compact outputs
result_top20, per_cmdb_anom

```
Out[10]:
```
Summary — network-related metrics during incident window (2022-03-20 21:00–21:30 UTC+8)

What I checked
- Focused on KPIs with network-related names (network, transmit, receive, packet, err, retransmit, rx/tx, tcp) including node-level series (container.node-5.*, container.node-6.*).
- Computed global p95/p5 on full series, then measured window statistics and counted points exceeding those thresholds.
- Excluded constant series and returned the top 20 (cmdb_id, kpi_name) by total anomaly counts.

Key findings (compact)
- Network receive/transmit packet/MB metrics show widespread anomalies across many services and nodes (node-5 and node-6):
  - frontend2-0: high receive and transmit packet counts (avg ~1,143 rx pkt, max ~1,629; avg ~1,214 tx pkt, max ~1,739). Anomaly counts: several points above p95 and several below p5. Earliest anomaly = 21:00:00.
  - adservice2-0: elevated receive/transmit packet metrics (avg ~447 rx, ~299 tx; max ~593 and ~415). Earliest anomaly = 21:00:00.
  - cartservice-0 and cartservice-2: elevated receive/transmit packet metrics (avg ~526 rx, ~463 tx; anomalies present; earliest = 21:00:00).
  - currencyservice-0 and currencyservice-1: elevated rx/tx metrics with anomalies at window start.
  - paymentservice-0, productcatalogservice-0, recommendationservice-1, redis-cart-0, redis-cart2-0, shippingservice-0 and others also appear in the top-20 network-anomaly list with similar patterns (multiple points above/below global thresholds).
- Typical pattern in top results: 30 points in the window, with about 2–3 points above the global p95 and 2–3 points below the global p5 — i.e., short deviations at the start of the window combined with lower values elsewhere.
- Earliest anomalies are concentrated at or very near the window start (21:00:00) for most affected cmdb_ids. A few services show earliest anomalies slightly later (examples in the full list: redis-cart2-0 earliest ≈ 21:02; some checkoutservice entries appear later).

Aggregate view
- Distinct cmdb_ids with network-related anomalies include (examples from the results): frontend2-0, adservice2-0, frontend-0/1/2, cartservice-0/2, cartservice2-0, currencyservice-0/1, paymentservice-0, productcatalogservice-0, recommendationservice-1, redis-cart-0, redis-cart2-0, shippingservice-0, checkoutservice-* and several others. Most have earliest anomaly timestamps at 21:00:00+08 (the full list shows ~40+ cmdb_ids with recorded network anomalies).

Interpretation
- The incident exhibits a broad network-level signal affecting many services/nodes (not limited to a single pod). The anomalies are primarily packet/bytes transmitted/received and appear at the start of the incident window.
- Pattern suggests transient bursts or shifts in network traffic (spikes above p95) plus periods below normal (below p5), indicating possible network congestion, retransmits, packet drops or routing issues affecting many services simultaneously.

Recommended next actions
1. Inspect node-level network interfaces (node-5 and node-6), NIC errors, dropped packets, retransmits, and link utilization around 21:00.  
2. Check infrastructure/network devices (switches, routers, load balancers) for errors or configuration events at the same time.  
3. Correlate these network anomalies with frontend error_logs ("failed to retrieve...") and trace anomalies (frontend→cart/checkout duration/row_count) already observed — this will help confirm whether network degradation caused frontend failures.  
4. If available, pull TCP-level metrics (retransmits, RTOs), kernel logs (dmesg), and host network counters for root-cause (packet drops, buffer exhaustion, or link flap).

If you want, I can (a) show the full top-20 rows verbatim, (b) list every cmdb_id with its earliest anomaly timestamp, or (c) cross-correlate the exact anomaly timestamps with frontend logs/traces. Which would you like next?

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id                                           kpi_name  total_points   global_p95   global_p5  points_in_window  anomaly_count_above_p95  anomaly_count_below_p5 earliest_anomaly_ts_in_window  avg_value_in_window  max_value_in_window  min_value_in_window
0               frontend2-0  container.node-6.container_network_receive_pac...            30  1492.000000  772.000000                30                        3                       3     2022-03-20 21:00:00+08:00          1143.400000          1628.833333           609.500000
1              adservice2-0  container.node-6.container_network_receive_pac...            30   563.500000  297.350000                30                        3                       2     2022-03-20 21:00:00+08:00           447.466667           592.500000           272.500000
2              adservice2-0  container.node-6.container_network_transmit_pa...            30   369.500000  195.900000                30                        3                       2     2022-03-20 21:00:00+08:00           299.166667           414.833333           190.500000
3             cartservice-0  container.node-5.container_network_receive_pac...            30   602.000000  401.625000                30                        3                       2     2022-03-20 21:00:00+08:00           526.400000           657.000000           348.500000
4             cartservice-2  container.node-5.container_network_transmit_pa...            30   540.000000  391.225000                30                        3                       2     2022-03-20 21:00:00+08:00           463.183333           564.000000           385.500000
5         currencyservice-0  container.node-5.container_network_receive_pac...            30   873.000000  606.975000                30                        3                       2     2022-03-20 21:00:00+08:00           749.483333           889.000000           560.000000
6         currencyservice-0  container.node-5.container_network_transmit_pa...            30   845.500000  550.600000                30                        3                       2     2022-03-20 21:00:00+08:00           715.166667           860.000000           510.000000
7         currencyservice-1  container.node-5.container_network_receive_pac...            30   911.000000  587.650000                30                        3                       2     2022-03-20 21:00:00+08:00           749.050000           966.500000           569.000000
8            emailservice-0  container.node-5.container_network_receive_MB....            30     0.027713    0.021177                30                        3                       2     2022-03-20 21:00:00+08:00             0.042504             0.554668             0.019342
9            emailservice-0  container.node-5.container_network_transmit_pa...            30   225.000000  151.375000                30                        3                       2     2022-03-20 21:00:00+08:00           189.066667           300.500000           148.000000
10              frontend2-0  container.node-6.container_network_transmit_pa...            30  1576.500000  812.875000                30                        3                       2     2022-03-20 21:00:00+08:00          1214.194444          1739.166667           606.000000
11         paymentservice-0  container.node-5.container_network_receive_MB....            30     0.024718    0.018768                30                        2                       3     2022-03-20 21:00:00+08:00             0.039015             0.551978             0.018641
12         paymentservice-0  container.node-5.container_network_receive_pac...            30   269.725000  211.000000                30                        2                       3     2022-03-20 21:00:00+08:00           238.833333           358.000000           208.500000
13         paymentservice-0  container.node-5.container_network_transmit_pa...            30   192.000000  143.225000                30                        3                       2     2022-03-20 21:00:00+08:00           166.616667           268.000000           143.000000
14  productcatalogservice-0  container.node-5.container_network_receive_MB....            30     1.506244    0.845750                30                        3                       2     2022-03-20 21:00:00+08:00             1.092931             1.508836             0.804529
15  recommendationservice-1  container.node-5.container_network_receive_pac...            30   562.000000  451.800000                30                        3                       2     2022-03-20 21:00:00+08:00           504.066667           595.500000           431.500000
16  recommendationservice-1  container.node-5.container_network_transmit_pa...            30   478.500000  366.425000                30                        3                       2     2022-03-20 21:00:00+08:00           419.200000           503.500000           356.000000
17             redis-cart-0  container.node-5.container_network_transmit_pa...            30   533.500000  421.725000                30                        3                       2     2022-03-20 21:00:00+08:00           484.333333           534.000000           360.000000
18        shippingservice-0  container.node-5.container_network_transmit_pa...            30   267.000000  187.475000                30                        3                       2     2022-03-20 21:00:00+08:00           231.350000           267.000000           177.000000
19            redis-cart2-0  container.node-5.container_network_transmit_pa...            30   282.000000  199.850000                30                        3                       2     2022-03-20 21:02:00+08:00           253.816667           448.500000           183.000000,                     cmdb_id       earliest_anomaly_ts
0                frontend-0 2022-03-20 21:00:00+08:00
1           emailservice2-0 2022-03-20 21:00:00+08:00
2          paymentservice-0 2022-03-20 21:00:00+08:00
3            emailservice-1 2022-03-20 21:00:00+08:00
4            emailservice-0 2022-03-20 21:00:00+08:00
..                      ...                       ...
37           emailservice-2 2022-03-20 21:05:00+08:00
38       currencyservice2-0 2022-03-20 21:07:00+08:00
39        checkoutservice-0 2022-03-20 21:09:00+08:00
40        checkoutservice-1 2022-03-20 21:14:00+08:00
41  recommendationservice-0 2022-03-20 21:20:00+08:00

[42 rows x 2 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[11]:

# Reuse df_metric and tz if present from previous steps; else load and prepare
import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# ensure df_metric and dt_local exist
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

if 'dt_local' not in df_metric.columns:
    df_metric['dt_local'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure timezone correctness
    df_metric['dt_local'] = pd.to_datetime(df_metric['dt_local']).dt.tz_convert(tz)

# Build masks: kpi_name must contain node-5 or node-6 AND one of the error/network keywords
node_mask = df_metric['kpi_name'].astype(str).str.contains(r'node-(5|6)', case=False, regex=True)
keywords = r'err|error|retransmit|retrans|drop|dropped|packet|rx_err|tx_err|packet_loss|retransmits|retransmit'
kw_mask = df_metric['kpi_name'].astype(str).str.contains(keywords, case=False, regex=True)

mask = node_mask & kw_mask
df_neterr_full = df_metric[mask].copy()

# Incident window (UTC+8)
start_window = pd.Timestamp('2022-03-20 21:00:00', tz=tz)
end_window = pd.Timestamp('2022-03-20 21:30:00', tz=tz)  # exclusive

# If no matching rows, return empty frames
if df_neterr_full.shape[0] == 0:
    result_top20 = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','total_points','global_p95','global_p5',
        'points_in_window','anomaly_count_above_p95','anomaly_count_below_p5',
        'earliest_anomaly_ts_in_window','avg_value_in_window','max_value_in_window','min_value_in_window'
    ])
    node_agg = pd.DataFrame(columns=['node','earliest_anomaly_ts'])
else:
    # 1) Compute global thresholds on the full series for each (cmdb_id,kpi_name) BEFORE window filtering
    pairs_global = (
        df_neterr_full
        .groupby(['cmdb_id','kpi_name'])['value']
        .agg(
            total_points='count',
            global_p95=lambda s: s.quantile(0.95),
            global_p5 = lambda s: s.quantile(0.05)
        )
        .reset_index()
    )

    # 2) Filter to incident window (on the subset)
    df_w = df_neterr_full[(df_neterr_full['dt_local'] >= start_window) & (df_neterr_full['dt_local'] < end_window)].copy()

    # 3) Merge global thresholds into window rows
    df_w = df_w.merge(pairs_global, on=['cmdb_id','kpi_name'], how='left', validate='m:1')

    # 4) Mark anomalies using global thresholds
    df_w['anom_above'] = df_w['value'] >= df_w['global_p95']
    df_w['anom_below'] = df_w['value'] <= df_w['global_p5']
    df_w['is_anom'] = df_w['anom_above'] | df_w['anom_below']

    # 5) Aggregate window stats per pair
    grp = df_w.groupby(['cmdb_id','kpi_name'])
    window_stats = grp.agg(
        points_in_window = ('value','size'),
        anomaly_count_above_p95 = ('anom_above','sum'),
        anomaly_count_below_p5 = ('anom_below','sum'),
        avg_value_in_window = ('value','mean'),
        max_value_in_window = ('value','max'),
        min_value_in_window = ('value','min')
    ).reset_index()

    # earliest anomaly timestamp per pair (where is_anom True)
    df_w_anom = df_w[df_w['is_anom']].groupby(['cmdb_id','kpi_name'])['dt_local'].min().reset_index().rename(columns={'dt_local':'earliest_anomaly_ts_in_window'})
    window_stats = window_stats.merge(df_w_anom, on=['cmdb_id','kpi_name'], how='left', validate='1:1')

    # 6) Combine global and window stats (only pairs that have points in window)
    combined = pairs_global.merge(window_stats, on=['cmdb_id','kpi_name'], how='inner', validate='1:1')

    # 7) Exclude constant series (max == min)
    combined_nonconst = combined[combined['max_value_in_window'] != combined['min_value_in_window']].copy()

    # 8) Compute anomaly_sum and sort as requested
    combined_nonconst['anomaly_sum'] = combined_nonconst['anomaly_count_above_p95'].fillna(0).astype(int) + combined_nonconst['anomaly_count_below_p5'].fillna(0).astype(int)
    combined_nonconst = combined_nonconst.sort_values(by=['anomaly_sum','earliest_anomaly_ts_in_window'], ascending=[False, True])

    # 9) Select top 20 and desired columns
    final_cols = [
        'cmdb_id','kpi_name','total_points','global_p95','global_p5',
        'points_in_window','anomaly_count_above_p95','anomaly_count_below_p5',
        'earliest_anomaly_ts_in_window','avg_value_in_window','max_value_in_window','min_value_in_window'
    ]
    result_top20 = combined_nonconst[final_cols].head(20).reset_index(drop=True)

    # 10) Per-node aggregate: detect node (node-5 or node-6) from kpi_name and report earliest anomaly ts per node if any anomaly_sum>0
    def extract_node(k):
        m = re.search(r'node-(5|6)', k, flags=re.IGNORECASE)
        return m.group(0).lower() if m else None

    combined_nonconst['node'] = combined_nonconst['kpi_name'].apply(extract_node)
    per_node = combined_nonconst[combined_nonconst['anomaly_sum']>0].groupby('node', as_index=False).agg(
        earliest_anomaly_ts = ('earliest_anomaly_ts_in_window','min')
    ).dropna(subset=['node'])

    # Ensure timezone aware and sort
    if not per_node.empty:
        per_node['earliest_anomaly_ts'] = pd.to_datetime(per_node['earliest_anomaly_ts']).dt.tz_convert(tz)
        node_agg = per_node.sort_values('earliest_anomaly_ts').reset_index(drop=True)
    else:
        node_agg = pd.DataFrame(columns=['node','earliest_anomaly_ts'])

# Compact outputs
result_top20, node_agg

```
Out[11]:
```
Summary — node-5 / node-6 network-error KPIs during incident window (2022-03-20 21:00–21:30 UTC+8)

What I did
- Selected metric KPIs whose name includes node-5 or node-6 and network/error keywords (err, retransmit, drop, packet, rx/tx, etc.).
- Computed global p95/p5 from the full series and measured window stats (counts, avg/max/min, anomaly counts relative to global thresholds) for each (cmdb_id, kpi_name).
- Excluded constant series and returned the top 20 non-constant pairs ranked by total anomaly counts.

Top observations (examples from the top-20)
- frontend2-0 / container.node-6.container_network_receive_packets
  - total_points: 30; window avg ≈ 1143.4, max ≈ 1628.8, min ≈ 609.5
  - anomalies: 3 points >= global_p95 and 3 points <= global_p5
  - earliest anomaly: 2022-03-20 21:00:00+08
- frontend2-0 / container.node-6.container_network_transmit_packets
  - avg ≈ 1214.2, max ≈ 1739.2; anomalies: 3 above / 2 below; earliest 21:00:00
- adservice2-0 / container.node-6 network rx/tx packets
  - avg rx ≈ 447.5 (max ≈ 592.5), avg tx ≈ 299.2 (max ≈ 414.8); both show 3 above / ~2 below p-thresholds; earliest 21:00:00
- cartservice-0, cartservice-2, currencyservice-0/1, recommendationservice-1, redis-cart-0, shippingservice-0, paymentservice-0, emailservice-* and others
  - show similar patterns: ~30 points in window, typically 2–3 points above global p95 and 2–3 below global p5, earliest anomalies at or near 21:00:00
- Some services show later earliest anomalies (example in full output: redis-cart2-0 earliest ≈ 21:02), but most start at 21:00.

Node-level aggregate
- Nodes with network-error anomalies: node-5 and node-6
- Earliest anomaly timestamp for each: node-5 → 2022-03-20 21:00:00+08, node-6 → 2022-03-20 21:00:00+08

Interpretation
- A broad, network-related signal affects many services and containers running on node-5 and node-6 at the start of the incident window. The pattern (a few samples above global p95 plus a few below p5) suggests transient spikes and drops in packet/byte counts (possible bursts, retransmits or packet loss) rather than steady degradation.
- The coincidence of earliest anomalies at ~21:00:00 across many services and both nodes points to an infrastructure/network-layer event (e.g., link congestion, NIC issues, switch/load-balancer event, or routing change) rather than a single application bug.

Recommended next steps
1. Inspect host/network counters on node-5 and node-6 at ~21:00 (NIC interface errors, drops, retransmits, interrupts, queue/full indicators).  
2. Check network infrastructure (switches, load balancers, routers) and any network change events/deploys around 21:00.  
3. Correlate these timestamps with frontend error logs and trace anomalies (frontend "failed to retrieve..." and frontend→checkout/cart trace hits) to confirm network-induced failures.  
4. If available, collect TCP-level retransmit metrics, kernel dmesg, and interface-level logs for root-cause.

If you want, I can (a) show the full top-20 table rows again, (b) list every affected kpi_name per node, or (c) cross-correlate these exact timestamps with frontend logs/traces. Which would you like next?

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id                                           kpi_name  total_points  global_p95  global_p5  points_in_window  anomaly_count_above_p95  anomaly_count_below_p5 earliest_anomaly_ts_in_window  avg_value_in_window  max_value_in_window  min_value_in_window
0               frontend2-0  container.node-6.container_network_receive_pac...            30    1492.000    772.000                30                        3                       3     2022-03-20 21:00:00+08:00          1143.400000          1628.833333                609.5
1              adservice2-0  container.node-6.container_network_receive_pac...            30     563.500    297.350                30                        3                       2     2022-03-20 21:00:00+08:00           447.466667           592.500000                272.5
2              adservice2-0  container.node-6.container_network_transmit_pa...            30     369.500    195.900                30                        3                       2     2022-03-20 21:00:00+08:00           299.166667           414.833333                190.5
3             cartservice-0  container.node-5.container_network_receive_pac...            30     602.000    401.625                30                        3                       2     2022-03-20 21:00:00+08:00           526.400000           657.000000                348.5
4             cartservice-2  container.node-5.container_network_transmit_pa...            30     540.000    391.225                30                        3                       2     2022-03-20 21:00:00+08:00           463.183333           564.000000                385.5
5         currencyservice-0  container.node-5.container_network_receive_pac...            30     873.000    606.975                30                        3                       2     2022-03-20 21:00:00+08:00           749.483333           889.000000                560.0
6         currencyservice-0  container.node-5.container_network_transmit_pa...            30     845.500    550.600                30                        3                       2     2022-03-20 21:00:00+08:00           715.166667           860.000000                510.0
7         currencyservice-1  container.node-5.container_network_receive_pac...            30     911.000    587.650                30                        3                       2     2022-03-20 21:00:00+08:00           749.050000           966.500000                569.0
8            emailservice-0  container.node-5.container_network_transmit_pa...            30     225.000    151.375                30                        3                       2     2022-03-20 21:00:00+08:00           189.066667           300.500000                148.0
9               frontend2-0  container.node-6.container_network_transmit_pa...            30    1576.500    812.875                30                        3                       2     2022-03-20 21:00:00+08:00          1214.194444          1739.166667                606.0
10         paymentservice-0  container.node-5.container_network_receive_pac...            30     269.725    211.000                30                        2                       3     2022-03-20 21:00:00+08:00           238.833333           358.000000                208.5
11         paymentservice-0  container.node-5.container_network_transmit_pa...            30     192.000    143.225                30                        3                       2     2022-03-20 21:00:00+08:00           166.616667           268.000000                143.0
12  recommendationservice-1  container.node-5.container_network_receive_pac...            30     562.000    451.800                30                        3                       2     2022-03-20 21:00:00+08:00           504.066667           595.500000                431.5
13  recommendationservice-1  container.node-5.container_network_transmit_pa...            30     478.500    366.425                30                        3                       2     2022-03-20 21:00:00+08:00           419.200000           503.500000                356.0
14             redis-cart-0  container.node-5.container_network_transmit_pa...            30     533.500    421.725                30                        3                       2     2022-03-20 21:00:00+08:00           484.333333           534.000000                360.0
15        shippingservice-0  container.node-5.container_network_transmit_pa...            30     267.000    187.475                30                        3                       2     2022-03-20 21:00:00+08:00           231.350000           267.000000                177.0
16            redis-cart2-0  container.node-5.container_network_transmit_pa...            30     282.000    199.850                30                        3                       2     2022-03-20 21:02:00+08:00           253.816667           448.500000                183.0
17           emailservice-1  container.node-5.container_network_transmit_pa...            30     221.925    154.675                30                        2                       2     2022-03-20 21:00:00+08:00           187.183333           228.000000                147.0
18          emailservice2-0  container.node-5.container_network_receive_pac...            30     327.300    234.700                30                        2                       2     2022-03-20 21:00:00+08:00           268.033333           357.000000                231.0
19          emailservice2-0  container.node-5.container_network_transmit_pa...            30     241.025    158.225                30                        2                       2     2022-03-20 21:00:00+08:00           183.483333           266.000000                151.5,      node       earliest_anomaly_ts
0  node-5 2022-03-20 21:00:00+08:00
1  node-6 2022-03-20 21:00:00+08:00)```
```